# Persian Emotional TTS
## نوت‌بوک مرحله‌ای پروژه در Google Colab

این نوت‌بوک برای تولید گفتار فارسی با کنترل لحن و احساس، بر اساس نقشه راه پنج‌فازی پروژه تنظیم شده است.

- **محیط اجرا:** Google Colab
- **ذخیره‌سازی پایدار:** Google Drive
- **مسیر پروژه:** `/content/drive/MyDrive/Persian_Emotional_TTS`
- **وضعیت فعلی:** کد گام‌های ۱.۱ تا ۴.۲ در نوت‌بوک موجود است؛ اتصال ONNX و WebSocket در گام ۴.۳ انجام می‌شود.

> سلول‌ها را به ترتیب اجرا کن. علامت کدِ موجود به معنی اجرای موفق یا تأیید کیفیت نیست؛ خروجی هر گام را جداگانه بررسی کن.

## نقشه راه پروژه

> علامت `[x]` یعنی کد آن گام در نوت‌بوک موجود است؛ اجرای موفق و کنترل کیفیت در Colab جداگانه انجام می‌شود.

### فاز ۱: زیرساخت، ابزارها و آماده‌سازی داده
- [x] **۱.۱:** راه‌اندازی Colab، اتصال Drive و ساختار پوشه‌بندی
- [x] **۱.۲:** نصب کتابخانه‌های کلیدی AI، پردازش صوت و مدل‌سازی
- [x] **۱.۳:** جمع‌آوری پیکره داده‌های داستانی و صوتی
- [x] **۱.۴:** پیش‌پردازش متن فارسی (G2P) و Force Alignment
- [x] **۱.۵:** برچسب‌گذاری احساسی اولیه (Pseudo-Labeling)

### فاز ۲: ماژول درک متن و تحلیل احساس
- [x] **۲.۱:** انتخاب مدل زبانی سبک فارسی
- [x] **۲.۲:** استخراج بردارهای Emotion/Prosody از متن

### فاز ۳: مدل آکوستیک و ووکودر
- [x] **۳.۱:** پیاده‌سازی Flow Matching مشابه Matcha-TTS
- [x] **۳.۲:** آموزش مدل آکوستیک با شرطی‌سازی احساس
- [x] **۳.۳:** اتصال BigVGAN برای تولید صوت

### فاز ۴: اجرای بلادرنگ و استریم
- [x] **۴.۱:** Sentence/Clause Chunking
- [x] **۴.۲:** Async Pipeline برای کاهش تأخیر
- [ ] **۴.۳:** ONNX و WebSocket Audio Streaming

### فاز ۵: ارزیابی و بهبود
- [ ] **۵.۱:** ارزیابی انسانی و MOS
- [ ] **۵.۲:** بررسی پایداری لحن در جملات طولانی و رفع اشکال

## قبل از اجرا

1. نوت‌بوک را در [Google Colab](https://colab.research.google.com/) باز کن.
2. از مسیر **Runtime → Change runtime type → Hardware accelerator** گزینه **GPU** را انتخاب کن.
3. سلول‌ها را از بالا به پایین اجرا کن.
4. هنگام اتصال Google Drive، دسترسی لازم را تأیید کن.
5. پس از اجرای هر گام، خروجی سلول پایانی همان گام را برای بررسی نگه دار.

# فاز ۱ — زیرساخت، ابزارها و آماده‌سازی داده
## گام ۱.۱ — بررسی محیط، اتصال Drive و ساختار پروژه

### سلول ۱: بررسی Python، PyTorch و GPU

این سلول نصب جدیدی انجام نمی‌دهد و فقط وضعیت runtime فعلی Colab را گزارش می‌کند.

In [ ]:
import platform
import sys
from pathlib import Path

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

try:
    import torch

    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("PyTorch CUDA runtime:", torch.version.cuda)
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
    else:
        print("WARNING: Select a GPU from Runtime > Change runtime type.")
except ImportError:
    print("PyTorch is not installed; it will be handled in step 1.2.")

### سلول ۲: اتصال Google Drive

در اولین اجرا، Colab پنجره اجازه دسترسی به Drive را نمایش می‌دهد. حساب موردنظر را انتخاب و دسترسی را تأیید کن.

In [ ]:
from google.colab import drive
from pathlib import Path

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")
else:
    print("Google Drive is already mounted.")

assert Path("/content/drive/MyDrive").is_dir(), "Google Drive mount failed."
print("Google Drive is ready.")

### سلول ۳: ساخت پوشه‌ها و فایل پیکربندی

اجرای این سلول امن و قابل تکرار است؛ فایل‌های صوتی یا مدل‌های موجود حذف نمی‌شوند.

In [ ]:
import json
from pathlib import Path

PROJECT_NAME = "Persian_Emotional_TTS"
PROJECT_ROOT = Path("/content/drive/MyDrive") / PROJECT_NAME
DIRECTORIES = (
    "data/raw",
    "data/processed",
    "models/acoustic",
    "models/emotion_extractor",
    "models/vocoder",
    "configs",
)

for relative_path in DIRECTORIES:
    (PROJECT_ROOT / relative_path).mkdir(parents=True, exist_ok=True)
    print(f"Ready: {relative_path}/")

project_config = {
    "project_name": PROJECT_NAME,
    "project_root": str(PROJECT_ROOT),
    "directories": list(DIRECTORIES),
}
config_path = PROJECT_ROOT / "configs" / "project_config.json"
config_path.write_text(
    json.dumps(project_config, ensure_ascii=False, indent=2),
    encoding="utf-8",
)


### سلول ۴: اعتبارسنجی ساختار پروژه

این سلول وجود همه پوشه‌ها و فایل پیکربندی را بررسی می‌کند.

In [ ]:
missing_directories = [
    relative_path
    for relative_path in DIRECTORIES
    if not (PROJECT_ROOT / relative_path).is_dir()
]
assert not missing_directories, f"Missing directories: {missing_directories}"
assert config_path.is_file(), "Project configuration was not created."

print("Project root:", PROJECT_ROOT)
print("Structure verification: PASSED")
print("Step 1.1 setup: VERIFIED")

## گام ۱.۲ — نصب کتابخانه‌های کلیدی

### سلول ۵: نصب یا تکمیل کتابخانه‌ها

Colab معمولاً PyTorch متناسب با CUDA خودش را از قبل دارد؛ بنابراین نسخه CUDA آن را بی‌دلیل تغییر نمی‌دهیم. اگر PyTorch/Torchaudio/Torchvision موجود نباشند، این سلول آن‌ها را هم نصب می‌کند.

کتابخانه‌های اختصاصی Matcha-TTS و BigVGAN عمداً در این مرحله نصب نمی‌شوند تا در فاز سوم، نسخه‌های سازگار با معماری پروژه انتخاب شوند.

In [ ]:
import importlib.util
import subprocess
import sys

packages = [
    "transformers",
    "datasets",
    "accelerate",
    "librosa",
    "soundfile",
    "numpy",
    "matplotlib",
    "tensorboard",
]

for package in ("torch", "torchaudio", "torchvision"):
    if importlib.util.find_spec(package) is None:
        packages.append(package)

print("Installing/checking:", ", ".join(packages))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", *packages],
    check=True,
)
print("Package installation completed.")

### سلول ۶: تست import و نسخه‌های کتابخانه‌ها

اجرای این سلول تایید می‌کند که همه کتابخانه‌های موردنیاز گام ۱.۲ بدون خطا import می‌شوند و در صورت وجود GPU، یک تست ساده CUDA نیز انجام می‌شود.

In [ ]:
import importlib
from importlib.metadata import PackageNotFoundError, version

packages_to_verify = (
    "torch",
    "torchaudio",
    "torchvision",
    "transformers",
    "datasets",
    "accelerate",
    "librosa",
    "soundfile",
    "numpy",
    "matplotlib",
    "tensorboard",
)

failed_imports = []
for package in packages_to_verify:
    try:
        importlib.import_module(package)
        try:
            package_version = version(package)
        except PackageNotFoundError:
            package_version = "version unavailable"
        print(f"OK  {package:<14} {package_version}")
    except Exception as exc:
        failed_imports.append(package)
        print(f"FAIL {package}: {type(exc).__name__}: {exc}")

if failed_imports:
    raise RuntimeError(
        "Step 1.2 is not complete. Failed imports: " + ", ".join(failed_imports)
    )

import torch

print("\nCUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    test_tensor = torch.rand(1024, device="cuda")
    print("CUDA tensor test: PASSED; mean =", round(test_tensor.mean().item(), 4))
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: Imports passed, but no GPU is active in this runtime.")

print("\nEnvironment verification: PASSED")

## گام ۱.۲ — اجرا و تأیید شد

کتابخانه‌های موردنیاز نصب و import آن‌ها بررسی شده است. اکنون سلول‌های گام ۱.۳ را ادامه بده. داده‌های پروژه همچنان روی Google Drive باقی می‌مانند.

## گام ۱.۳ — دانلود و آماده‌سازی پیکره پایه فارسی

در این گام از **Google FLEURS** با زبان و کانفیگ `fa_ir` استفاده می‌کنیم. این دیتاست برای راه‌اندازی و تست اولیه مدل آکوستیک مناسب است و نیازی به آپلود دستی فایل‌ها ندارد.

**نکته‌های مهم:**

- FLEURS دیتاست گفتار خوانده‌شده و پایه است؛ **دیتاست داستان‌گویی احساسی نیست**. داده داستانی و احساسی در مراحل بعد به ساختار پروژه اضافه می‌شود.
- صوت اصلی FLEURS معمولاً **۱۶۰۰۰ هرتز** است. خروجی این نوت‌بوک برای یکسان‌سازی خط پردازش، به **WAV مونو با ۲۲۰۵۰ هرتز** تبدیل می‌شود؛ resample کردن به‌خودی‌خود جزئیات ازدست‌رفته را برنمی‌گرداند.
- متن خام و علائم آن تا گام ۱.۴ حفظ می‌شود؛ در این مرحله نرمال‌سازی متن فارسی انجام نمی‌دهیم.
- برای دانلود، Colab باید به اینترنت و Hugging Face دسترسی داشته باشد.

### سلول ۷: تنظیمات و دانلود FLEURS فارسی

برای یک تست سریع می‌توانی مقدار `MAX_SAMPLES_PER_SPLIT` را مثلاً `20` بگذاری. مقدار `None` یعنی پردازش همه نمونه‌های splitهای انتخاب‌شده.

این سلول فقط دیتاست را دانلود و در cache کتابخانه `datasets` قرار می‌دهد؛ تبدیل فایل‌ها در سلول بعد انجام می‌شود.

In [ ]:
import json
from pathlib import Path

from datasets import load_dataset

PROJECT_ROOT = Path("/content/drive/MyDrive/Persian_Emotional_TTS")
assert Path("/content/drive/MyDrive").is_dir(), "Run the Google Drive mounting cell first."
assert PROJECT_ROOT.is_dir(), "Project root not found; run the step 1.1 setup cells first."

DATASET_NAME = "google/fleurs"
DATASET_CONFIG = "fa_ir"
DATASET_ROOT = PROJECT_ROOT / "data" / "raw" / "fleurs_fa_ir"
SPLITS = ("train", "validation", "test")
MAX_SAMPLES_PER_SPLIT = None  # Use an integer such as 20 for a quick test.
OUTPUT_SAMPLE_RATE = 22050
OVERWRITE_EXISTING = False

DATASET_ROOT.mkdir(parents=True, exist_ok=True)
for folder in ("audio", "transcripts"):
    (DATASET_ROOT / folder).mkdir(parents=True, exist_ok=True)

print(f"Loading {DATASET_NAME} ({DATASET_CONFIG})...")
fleurs = load_dataset(DATASET_NAME, DATASET_CONFIG)
print(fleurs)

for split in SPLITS:
    assert split in fleurs, f"Split {split!r} is not available in FLEURS."

run_config = {
    "dataset_name": DATASET_NAME,
    "dataset_config": DATASET_CONFIG,
    "splits": list(SPLITS),
    "max_samples_per_split": MAX_SAMPLES_PER_SPLIT,
    "output_sample_rate": OUTPUT_SAMPLE_RATE,
    "output_format": "WAV PCM_16 mono",
    "text_field_preference": ["raw_transcription", "transcription"],
}
(DATASET_ROOT / "dataset_config.json").write_text(
    json.dumps(run_config, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("Dataset download and configuration: READY")

### سلول ۸: تبدیل صوت‌ها و ساخت فایل‌های متن و metadata

خروجی در مسیر زیر ذخیره می‌شود:

```text
data/raw/fleurs_fa_ir/
├── audio/{train,validation,test}/
├── transcripts/{train,validation,test}/
├── metadata.csv
└── dataset_config.json
```

فایل‌ها نام‌های یکتا و قابل تکرار دارند. اجرای دوباره سلول، فایل‌های موجود را به‌صورت پیش‌فرض بازنویسی نمی‌کند. برای بازنویسی، `OVERWRITE_EXISTING = True` را در سلول قبل تنظیم کن.

In [ ]:
import csv
import re
from pathlib import Path

import librosa
import numpy as np
import soundfile as sf
from tqdm.auto import tqdm


def as_mono(audio_array):
    """Convert decoded audio to a one-dimensional float32 mono array."""
    array = np.asarray(audio_array, dtype=np.float32)
    if array.ndim == 1:
        return array
    if array.ndim == 2:
        # Handle either (channels, samples) or (samples, channels).
        channel_axis = 0 if array.shape[0] <= array.shape[1] else 1
        return array.mean(axis=channel_axis, dtype=np.float32)
    raise ValueError(f"Unexpected audio shape: {array.shape}")


rows = []
for split in SPLITS:
    split_data = fleurs[split]
    if MAX_SAMPLES_PER_SPLIT is not None:
        sample_count = min(MAX_SAMPLES_PER_SPLIT, len(split_data))
        split_data = split_data.select(range(sample_count))

    for index, row in enumerate(tqdm(split_data, desc=f"Preparing {split}")):
        source_id = str(row.get("id", index))
        safe_source_id = re.sub(r"[^A-Za-z0-9_-]+", "_", source_id)
        utterance_id = f"fleurs_fa_ir_{split}_{index:06d}_{safe_source_id}"

        audio_path = DATASET_ROOT / "audio" / split / f"{utterance_id}.wav"
        transcript_path = DATASET_ROOT / "transcripts" / split / f"{utterance_id}.txt"
        audio_path.parent.mkdir(parents=True, exist_ok=True)
        transcript_path.parent.mkdir(parents=True, exist_ok=True)

        text = str(row.get("raw_transcription") or row.get("transcription") or "").strip()
        if not text:
            raise ValueError(f"Empty transcription for {split} sample {index}.")

        audio_data = row["audio"]
        if not audio_data or "array" not in audio_data:
            raise ValueError(f"Decoded audio is unavailable for {utterance_id}.")
        source_sample_rate = int(audio_data["sampling_rate"])
        audio_array = as_mono(audio_data["array"])
        if audio_array.size == 0 or source_sample_rate <= 0:
            raise ValueError(f"Empty or invalid audio for {utterance_id}.")

        if source_sample_rate != OUTPUT_SAMPLE_RATE:
            audio_array = librosa.resample(
                audio_array,
                orig_sr=source_sample_rate,
                target_sr=OUTPUT_SAMPLE_RATE,
                res_type="kaiser_best",
            )
        audio_array = np.clip(audio_array, -1.0, 1.0)

        if OVERWRITE_EXISTING or not audio_path.exists():
            sf.write(str(audio_path), audio_array, OUTPUT_SAMPLE_RATE, subtype="PCM_16")
        if OVERWRITE_EXISTING or not transcript_path.exists():
            transcript_path.write_text(text + "\n", encoding="utf-8")

        rows.append({
            "utterance_id": utterance_id,
            "audio_path": audio_path.relative_to(PROJECT_ROOT).as_posix(),
            "transcript_path": transcript_path.relative_to(PROJECT_ROOT).as_posix(),
            "text": text,
            "speaker_id": str(row.get("speaker_id", "unknown")),
            "source": DATASET_NAME,
            "source_config": DATASET_CONFIG,
            "source_id": source_id,
            "split": split,
            "duration_seconds": round(len(audio_array) / OUTPUT_SAMPLE_RATE, 3),
            "sample_rate": OUTPUT_SAMPLE_RATE,
            "source_sample_rate": source_sample_rate,
            "channels": 1,
        })

metadata_path = DATASET_ROOT / "metadata.csv"
fieldnames = (
    "utterance_id", "audio_path", "transcript_path", "text", "speaker_id",
    "source", "source_config", "source_id", "split", "duration_seconds",
    "sample_rate", "source_sample_rate", "channels",
)
with metadata_path.open("w", encoding="utf-8", newline="") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=fieldnames, lineterminator="\n")
    writer.writeheader()
    writer.writerows(rows)

print("Prepared utterances:", len(rows))
print("Total duration (minutes):", round(sum(row["duration_seconds"] for row in rows) / 60, 2))
print("Metadata:", metadata_path)
print("FLEURS audio and transcript export: COMPLETE")

### سلول ۹: اعتبارسنجی مستقل خروجی FLEURS

این سلول فایل `metadata.csv` را از روی Drive می‌خواند و وجود صوت و متن، مونو بودن، نرخ ۲۲۰۵۰ هرتز و غیرخالی بودن متن‌ها را بررسی می‌کند. نتیجه آن را برای ورود به گام ۱.۴ ارسال کن.

In [ ]:
import csv
from collections import Counter

import soundfile as sf

assert metadata_path.is_file(), "metadata.csv was not created; rerun the export cell."
with metadata_path.open("r", encoding="utf-8", newline="") as csv_file:
    manifest_rows = list(csv.DictReader(csv_file))

assert manifest_rows, "The FLEURS manifest is empty."
validation_errors = []
for row in manifest_rows:
    audio_path = PROJECT_ROOT / row["audio_path"]
    transcript_path = PROJECT_ROOT / row["transcript_path"]
    try:
        assert audio_path.is_file(), f"Missing audio: {audio_path}"
        assert transcript_path.is_file(), f"Missing transcript: {transcript_path}"
        info = sf.info(str(audio_path))
        assert info.samplerate == OUTPUT_SAMPLE_RATE, (
            f"Unexpected sample rate for {row['utterance_id']}: {info.samplerate}"
        )
        assert info.channels == 1, f"Not mono: {row['utterance_id']}"
        assert info.frames > 0, f"Empty audio: {row['utterance_id']}"
        transcript = transcript_path.read_text(encoding="utf-8").strip()
        assert transcript, f"Empty transcript: {row['utterance_id']}"
        assert transcript == row["text"].strip(), f"Text mismatch: {row['utterance_id']}"
    except Exception as exc:
        validation_errors.append(f"{row['utterance_id']}: {exc}")

if validation_errors:
    raise ValueError("FLEURS validation failed:\n" + "\n".join(validation_errors[:20]))

split_counts = Counter(row["split"] for row in manifest_rows)
total_seconds = sum(float(row["duration_seconds"]) for row in manifest_rows)
print("Validated utterances:", len(manifest_rows))
print("Rows per split:", dict(split_counts))
print("Total duration (minutes):", round(total_seconds / 60, 2))
print("Output sample rate:", OUTPUT_SAMPLE_RATE)
print("FLEURS fa_ir preparation: PASSED")

## گام ۱.۳ — پیکره پایه آماده شد

پس از اجرای موفق سلول‌های ۷ تا ۹، دیتاست پایه FLEURS فارسی در Drive قرار دارد و می‌توانیم از آن برای تست خط پیش‌پردازش و آموزش اولیه استفاده کنیم.

این خروجی به‌تنهایی مدل را داستان‌گو یا احساسی نمی‌کند. در ادامه به داده داستانی و احساسی، یا ضبط اختصاصی با متن هماهنگ، نیاز داریم. گام بعدی **۱.۴: نرمال‌سازی متن فارسی، G2P و Force Alignment** است.

## گام ۱.۴ — پیش‌پردازش متن فارسی، G2P و Force Alignment

در این گام دو خروجی جدا ساخته می‌شود:

1. **متن نرمال‌شده و IPA:** با `hazm` و `eSpeak-NG` از طریق `phonemizer`.
2. **همگام‌سازی کلمه‌ای:** با `stable-ts` و استفاده از متن مرجع هر فایل، نه تولید متن جدید توسط ASR.

G2P در این مرحله یک baseline قابل اجرا برای فارسی است و بعداً می‌توان آن را با مدل تخصصی‌تر فارسی جایگزین کرد. کیفیت alignment نیز باید روی نمونه‌هایی از داده بررسی شود؛ اجرای بدون خطا به‌تنهایی اثبات دقت تمام مرزهای کلمات نیست.

### سلول ۱۰: نصب ابزارهای G2P و alignment

این سلول `espeak-ng` و `ffmpeg` را در محیط Colab نصب می‌کند و سپس کتابخانه‌های Python لازم را نصب می‌نماید. اجرای آن فقط در Colab لازم است.

In [ ]:
import subprocess
import sys

subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "espeak-ng", "ffmpeg"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "hazm", "phonemizer", "stable-ts"],
    check=True,
)
print("G2P and alignment dependencies: READY")

### سلول ۱۱: نرمال‌سازی متن و تولید IPA

متن خام `metadata.csv` تغییر نمی‌کند. خروجی این سلول در `data/processed/fleurs_fa_ir/metadata_text.csv` ذخیره می‌شود و شامل `normalized_text` و `phonemes` است.

In [ ]:
import csv
import json
from pathlib import Path

import hazm
from phonemizer import phonemize
from phonemizer.separator import Separator

PROJECT_ROOT = Path("/content/drive/MyDrive/Persian_Emotional_TTS")
RAW_METADATA_PATH = PROJECT_ROOT / "data" / "raw" / "fleurs_fa_ir" / "metadata.csv"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed" / "fleurs_fa_ir"
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

assert RAW_METADATA_PATH.is_file(), "Run the FLEURS preparation cells first."
with RAW_METADATA_PATH.open("r", encoding="utf-8", newline="") as csv_file:
    raw_rows = list(csv.DictReader(csv_file))
assert raw_rows, "The FLEURS metadata is empty."

normalizer = hazm.Normalizer()
phone_separator = Separator(phone=" ", word=" | ")
text_rows = []
g2p_errors = []

for row in raw_rows:
    try:
        normalized_text = " ".join(normalizer.normalize(row["text"]).split())
        if not normalized_text:
            raise ValueError("Normalized text is empty.")
        phonemes = phonemize(
            normalized_text,
            language="fa",
            backend="espeak",
            separator=phone_separator,
            strip=True,
            preserve_punctuation=False,
            njobs=1,
        ).strip()
        if not phonemes:
            raise ValueError("G2P returned an empty phoneme sequence.")
        output_row = dict(row)
        output_row["normalized_text"] = normalized_text
        output_row["phonemes"] = phonemes
        text_rows.append(output_row)
    except Exception as exc:
        g2p_errors.append({"utterance_id": row["utterance_id"], "error": str(exc)})

if g2p_errors:
    (PROCESSED_ROOT / "g2p_errors.json").write_text(
        json.dumps(g2p_errors, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    raise RuntimeError(f"G2P failed for {len(g2p_errors)} rows; see g2p_errors.json")

text_metadata_path = PROCESSED_ROOT / "metadata_text.csv"
with text_metadata_path.open("w", encoding="utf-8", newline="") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=list(text_rows[0]), lineterminator="\n")
    writer.writeheader()
    writer.writerows(text_rows)

print("G2P processed utterances:", len(text_rows))
print("Text metadata:", text_metadata_path)
print("Sample normalized text:", text_rows[0]["normalized_text"])
print("Sample IPA:", text_rows[0]["phonemes"])
print("Persian text preprocessing: PASSED")

### سلول ۱۲: بارگذاری مدل alignment

مدل `base` برای اولین اجرا سبک‌تر است. اگر کیفیت مرزهای کلمات کافی نبود، می‌توان بعداً `small` را امتحان کرد؛ این تغییر سرعت و مصرف حافظه را بیشتر می‌کند.

`ALIGNMENT_LIMIT = 20` فقط برای Smoke Test است. برای پردازش همه ردیف‌ها، پس از موفقیت تست آن را به `None` تغییر بده.

In [ ]:
import torch
import stable_whisper

ALIGNMENT_MODEL_NAME = "base"
ALIGNMENT_LIMIT = 20  # Smoke test only; set to None for the full manifest.
OVERWRITE_ALIGNMENT = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Alignment device:", DEVICE)
print("Loading alignment model:", ALIGNMENT_MODEL_NAME)
alignment_model = stable_whisper.load_model(ALIGNMENT_MODEL_NAME, device=DEVICE)
print("Alignment model: READY")

### سلول ۱۳: اجرای Force Alignment روی متن مرجع

برای هر نمونه یک فایل JSON با timestamp کلمات ساخته می‌شود. فایل‌های موجود در اجرای بعدی خوانده می‌شوند، بنابراین بعد از تست ۲۰ نمونه‌ای می‌توانی با `ALIGNMENT_LIMIT = None` ادامه بدهی.

اگر متن و صوت مطابقت نداشته باشند یا مدل نتواند کلمات را align کند، خطا ثبت و گزارش می‌شود؛ نباید خروجی مشکوک را بدون بررسی برای آموزش استفاده کرد.

In [ ]:
import csv
import json
from tqdm.auto import tqdm

alignment_root = PROCESSED_ROOT / "alignments"
alignment_root.mkdir(parents=True, exist_ok=True)
aligned_metadata_path = PROCESSED_ROOT / "metadata_aligned.csv"

alignment_input_rows = text_rows if ALIGNMENT_LIMIT is None else text_rows[:ALIGNMENT_LIMIT]
aligned_rows = []
alignment_errors = []

for row in tqdm(alignment_input_rows, desc="Force aligning"):
    utterance_id = row["utterance_id"]
    alignment_path = alignment_root / f"{utterance_id}.json"
    try:
        if alignment_path.exists() and not OVERWRITE_ALIGNMENT:
            alignment_payload = json.loads(alignment_path.read_text(encoding="utf-8"))
        else:
            result = alignment_model.align(
                str(PROJECT_ROOT / row["audio_path"]),
                row["text"],
                language="fa",
                regroup=False,
            )
            result_dict = result.to_dict()
            words = []
            for segment in result_dict.get("segments", []):
                for word in segment.get("words", []) or []:
                    word_text = str(word.get("word", "")).strip()
                    if not word_text:
                        continue
                    words.append({
                        "word": word_text,
                        "start": float(word.get("start") or 0.0),
                        "end": float(word.get("end") or 0.0),
                    })
            if not words:
                raise ValueError("No word timestamps were returned.")
            alignment_payload = {
                "utterance_id": utterance_id,
                "audio_path": row["audio_path"],
                "text": row["text"],
                "normalized_text": row["normalized_text"],
                "phonemes": row["phonemes"],
                "language": "fa",
                "alignment_model": ALIGNMENT_MODEL_NAME,
                "segments": result_dict.get("segments", []),
                "words": words,
            }
            alignment_path.write_text(
                json.dumps(alignment_payload, ensure_ascii=False, indent=2),
                encoding="utf-8",
            )

        words = alignment_payload.get("words", [])
        if not words:
            raise ValueError("Alignment JSON contains no words.")
        output_row = dict(row)
        output_row["alignment_path"] = alignment_path.relative_to(PROJECT_ROOT).as_posix()
        output_row["alignment_word_count"] = len(words)
        aligned_rows.append(output_row)
    except Exception as exc:
        alignment_errors.append({"utterance_id": utterance_id, "error": str(exc)})

if alignment_errors:
    (PROCESSED_ROOT / "alignment_errors.json").write_text(
        json.dumps(alignment_errors, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    raise RuntimeError(
        f"Alignment failed for {len(alignment_errors)} rows; see alignment_errors.json"
    )

with aligned_metadata_path.open("w", encoding="utf-8", newline="") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=list(aligned_rows[0]), lineterminator="\n")
    writer.writeheader()
    writer.writerows(aligned_rows)

print("Aligned utterances:", len(aligned_rows))
print("Aligned metadata:", aligned_metadata_path)
print("Alignment JSON directory:", alignment_root)
print("Force alignment export: COMPLETE")

### سلول ۱۴: اعتبارسنجی timestampها و خروجی alignment

این سلول وجود فایل JSON، تعداد کلمات و معتبر بودن بازه‌های زمانی را بررسی می‌کند. اگر `ALIGNMENT_LIMIT` هنوز ۲۰ باشد، نتیجه فقط اعتبارسنجی یک زیرمجموعه آزمایشی است، نه پردازش کامل دیتاست.

In [ ]:
import csv
import json

assert aligned_metadata_path.is_file(), "Aligned metadata was not created; rerun the alignment cell."
with aligned_metadata_path.open("r", encoding="utf-8", newline="") as csv_file:
    aligned_manifest = list(csv.DictReader(csv_file))

expected_count = len(alignment_input_rows)
assert len(aligned_manifest) == expected_count, "Aligned row count does not match the requested subset."
alignment_validation_errors = []
word_count = 0

for row in aligned_manifest:
    try:
        payload_path = PROJECT_ROOT / row["alignment_path"]
        payload = json.loads(payload_path.read_text(encoding="utf-8"))
        words = payload["words"]
        assert words, f"No words in {row['utterance_id']}"
        assert len(words) == int(row["alignment_word_count"])
        previous_start = -1.0
        for word in words:
            start = float(word["start"])
            end = float(word["end"])
            assert start >= 0 and end > start, f"Invalid word span: {word}"
            assert start >= previous_start, f"Non-monotonic word timestamps: {row['utterance_id']}"
            previous_start = start
            word_count += 1
    except Exception as exc:
        alignment_validation_errors.append(f"{row['utterance_id']}: {exc}")

if alignment_validation_errors:
    raise ValueError("Alignment validation failed:\n" + "\n".join(alignment_validation_errors[:20]))

print("Validated aligned utterances:", len(aligned_manifest))
print("Validated word timestamps:", word_count)
print("Alignment subset limit:", ALIGNMENT_LIMIT)
print("Alignment output verification: PASSED")

## گام ۱.۴ — نتیجه و ادامه

با اجرای موفق سلول‌های ۱۰ تا ۱۴، مسیر G2P و alignment تست شده است. اگر مقدار `ALIGNMENT_LIMIT` را روی ۲۰ گذاشته‌ای، برای پردازش کامل دیتاست آن را به `None` تغییر بده و سلول‌های ۱۳ و ۱۴ را دوباره اجرا کن.

خروجی‌های اصلی:

```text
data/processed/fleurs_fa_ir/metadata_text.csv
data/processed/fleurs_fa_ir/metadata_aligned.csv
data/processed/fleurs_fa_ir/alignments/
```

گام بعدی، **۱.۵: Pseudo-Labeling احساسی**، در ادامه همین نوت‌بوک آماده است. قبل از استفاده از برچسب‌ها برای آموزش، نمونه‌های کم‌اطمینان را بازبینی انسانی کن.

## گام ۱.۵ — Pseudo-Labeling احساسی

در این گام از مدل فارسی [m3hrdadfi/wav2vec2-xlsr-persian-speech-emotion-recognition](https://huggingface.co/m3hrdadfi/wav2vec2-xlsr-persian-speech-emotion-recognition) برای تولید برچسب اولیه از روی صوت استفاده می‌کنیم.

**این برچسب‌ها ground truth نیستند.** خروجی مدل به‌همراه confidence و فاصله دو برچسب برتر (margin) ذخیره می‌شود و نمونه‌های کم‌اطمینان برای بازبینی انسانی علامت می‌خورند. مدل ممکن است روی FLEURS یا سبک داستان‌گویی خطا داشته باشد.

کلاس‌های استاندارد خروجی پروژه:

```text
neutral, happy, sad, angry, fearful, surprised
```

ورودی این گام `metadata_text.csv` از گام ۱.۴ است. صوت اصلی تغییر نمی‌کند و فقط در حافظه برای مدل به نرخ موردنیاز مدل resample می‌شود.

### سلول ۱۵: بارگذاری مدل احساس و تنظیم آستانه‌ها

برای تست سریع، `PSEUDO_LABEL_LIMIT = 20` است. بعد از بررسی موفق خروجی، برای پردازش همه ردیف‌ها آن را به `None` تغییر بده.

`EMOTION_CONFIDENCE_THRESHOLD` و `EMOTION_MARGIN_THRESHOLD` فقط برای ساخت صف بازبینی هستند؛ برچسب اصلی مدل حذف یا با `neutral` جایگزین نمی‌شود.

In [ ]:
import json
from pathlib import Path

import torch
from transformers import AutoConfig, AutoFeatureExtractor, AutoModelForAudioClassification

PROJECT_ROOT = Path("/content/drive/MyDrive/Persian_Emotional_TTS")
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed" / "fleurs_fa_ir"
TEXT_METADATA_PATH = PROCESSED_ROOT / "metadata_text.csv"
EMOTION_ROOT = PROCESSED_ROOT / "emotion"
EMOTION_ROOT.mkdir(parents=True, exist_ok=True)

EMOTION_MODEL_NAME = "m3hrdadfi/wav2vec2-xlsr-persian-speech-emotion-recognition"
PSEUDO_LABEL_LIMIT = 20  # Smoke test; set to None for the full text manifest.
EMOTION_CONFIDENCE_THRESHOLD = 0.60
EMOTION_MARGIN_THRESHOLD = 0.15
EMOTION_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CANONICAL_LABELS = ("neutral", "happy", "sad", "angry", "fearful", "surprised")
LABEL_ALIASES = {
    "neutral": "neutral", "neu": "neutral",
    "happiness": "happy", "happy": "happy", "hap": "happy",
    "sadness": "sad", "sad": "sad",
    "anger": "angry", "angry": "angry", "ang": "angry",
    "fear": "fearful", "fearful": "fearful", "fea": "fearful",
    "surprise": "surprised", "surprised": "surprised", "sur": "surprised",
}


def canonical_emotion_label(label):
    key = str(label).strip().lower().replace("-", "_").replace(" ", "_")
    if key not in LABEL_ALIASES:
        raise ValueError(f"Unsupported emotion label from model: {label!r}")
    return LABEL_ALIASES[key]


assert TEXT_METADATA_PATH.is_file(), "Run the Persian G2P cell (step 1.4) first."
print("Loading emotion model:", EMOTION_MODEL_NAME)
emotion_model_config = AutoConfig.from_pretrained(EMOTION_MODEL_NAME)
emotion_processor = AutoFeatureExtractor.from_pretrained(EMOTION_MODEL_NAME)
EMOTION_SAMPLE_RATE = int(getattr(emotion_processor, "sampling_rate", 16000))
emotion_model = AutoModelForAudioClassification.from_pretrained(EMOTION_MODEL_NAME)
emotion_model = emotion_model.to(EMOTION_DEVICE).eval()
MODEL_LABELS = {
    int(index): canonical_emotion_label(label)
    for index, label in emotion_model_config.id2label.items()
}

print("Emotion model device:", EMOTION_DEVICE)
print("Emotion model sample rate:", EMOTION_SAMPLE_RATE)
print("Canonical model labels:", MODEL_LABELS)
print("Pseudo-label limit:", PSEUDO_LABEL_LIMIT)

### سلول ۱۶: تولید برچسب اولیه و ساخت صف بازبینی

خروجی‌ها در `data/processed/fleurs_fa_ir/emotion/` ذخیره می‌شوند:

- `metadata_emotion.csv`: متن، مسیر صوت و برچسب اولیه برای هر نمونه
- `emotion_predictions.json`: احتمال تمام کلاس‌ها برای بررسی و استفاده بعدی
- `emotion_review.csv`: نمونه‌هایی که confidence یا margin آن‌ها از آستانه پایین‌تر است
- `emotion_config.json`: مدل و آستانه‌های استفاده‌شده

برای تست اولیه، ۲۰ نمونه کافی است؛ این عدد محدودیت پردازش است، نه معیار کافی‌بودن دیتاست برای آموزش نهایی.

In [ ]:
import csv
import json
from collections import Counter

import librosa
import torch
from tqdm.auto import tqdm

with TEXT_METADATA_PATH.open("r", encoding="utf-8", newline="") as csv_file:
    text_manifest_for_emotion = list(csv.DictReader(csv_file))
assert text_manifest_for_emotion, "The text manifest is empty."

emotion_input_rows = (
    text_manifest_for_emotion
    if PSEUDO_LABEL_LIMIT is None
    else text_manifest_for_emotion[:PSEUDO_LABEL_LIMIT]
)
assert emotion_input_rows, "No rows selected for pseudo-labeling."

emotion_rows = []
emotion_predictions = []
emotion_errors = []


def predict_emotion_probabilities(audio_path):
    speech, _ = librosa.load(
        str(audio_path), sr=EMOTION_SAMPLE_RATE, mono=True
    )
    if len(speech) == 0:
        raise ValueError(f"Empty audio: {audio_path}")
    model_inputs = emotion_processor(
        speech,
        sampling_rate=EMOTION_SAMPLE_RATE,
        padding=True,
        return_tensors="pt",
    )
    model_inputs = {
        key: value.to(EMOTION_DEVICE) for key, value in model_inputs.items()
    }
    with torch.inference_mode():
        logits = emotion_model(**model_inputs).logits
        probabilities = torch.softmax(logits, dim=-1)[0].detach().cpu().tolist()
    return {
        MODEL_LABELS[index]: float(score)
        for index, score in enumerate(probabilities)
    }


for row in tqdm(emotion_input_rows, desc="Pseudo-labeling emotions"):
    try:
        probabilities = predict_emotion_probabilities(PROJECT_ROOT / row["audio_path"])
        ranked_labels = sorted(probabilities.items(), key=lambda item: item[1], reverse=True)
        top_label, confidence = ranked_labels[0]
        margin = confidence - ranked_labels[1][1]
        review_required = bool(
            confidence < EMOTION_CONFIDENCE_THRESHOLD
            or margin < EMOTION_MARGIN_THRESHOLD
        )

        output_row = dict(row)
        output_row.update({
            "pseudo_emotion": top_label,
            "pseudo_confidence": round(confidence, 6),
            "pseudo_margin": round(margin, 6),
            "emotion_review_required": review_required,
            "emotion_model": EMOTION_MODEL_NAME,
            "emotion_label_source": "pseudo",
        })
        emotion_rows.append(output_row)
        emotion_predictions.append({
            "utterance_id": row["utterance_id"],
            "probabilities": probabilities,
            "pseudo_emotion": top_label,
            "confidence": confidence,
            "margin": margin,
            "review_required": review_required,
        })
    except Exception as exc:
        emotion_errors.append({"utterance_id": row["utterance_id"], "error": str(exc)})

if emotion_errors:
    (EMOTION_ROOT / "emotion_errors.json").write_text(
        json.dumps(emotion_errors, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    raise RuntimeError(
        f"Emotion inference failed for {len(emotion_errors)} rows; see emotion_errors.json"
    )

emotion_metadata_path = EMOTION_ROOT / "metadata_emotion.csv"
with emotion_metadata_path.open("w", encoding="utf-8", newline="") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=list(emotion_rows[0]), lineterminator="\n")
    writer.writeheader()
    writer.writerows(emotion_rows)

predictions_path = EMOTION_ROOT / "emotion_predictions.json"
predictions_path.write_text(
    json.dumps(emotion_predictions, ensure_ascii=False, indent=2), encoding="utf-8"
)

review_path = EMOTION_ROOT / "emotion_review.csv"
review_rows = [row for row in emotion_rows if row["emotion_review_required"]]
with review_path.open("w", encoding="utf-8", newline="") as csv_file:
    writer = csv.DictWriter(csv_file, fieldnames=list(emotion_rows[0]), lineterminator="\n")
    writer.writeheader()
    writer.writerows(review_rows)

emotion_run_config = {
    "model_name": EMOTION_MODEL_NAME,
    "sample_rate": EMOTION_SAMPLE_RATE,
    "limit": PSEUDO_LABEL_LIMIT,
    "confidence_threshold": EMOTION_CONFIDENCE_THRESHOLD,
    "margin_threshold": EMOTION_MARGIN_THRESHOLD,
    "label_source": "pseudo",
    "canonical_labels": list(CANONICAL_LABELS),
}
(EMOTION_ROOT / "emotion_config.json").write_text(
    json.dumps(emotion_run_config, ensure_ascii=False, indent=2), encoding="utf-8"
)

print("Pseudo-labeled utterances:", len(emotion_rows))
print("Label distribution:", dict(Counter(row["pseudo_emotion"] for row in emotion_rows)))
print("Review required:", len(review_rows))
print("Emotion metadata:", emotion_metadata_path)
print("Review queue:", review_path)
print("Pseudo-label export: COMPLETE")

### سلول ۱۷: اعتبارسنجی برچسب‌ها و احتمال‌ها

این سلول برچسب‌های مجاز، بازه confidence و margin، وجود احتمال برای هر نمونه و مجموع احتمال‌ها را بررسی می‌کند. موفقیت این سلول به معنی صحت معنایی برچسب احساس نیست؛ برای استفاده جدی در آموزش، بخشی از صف بازبینی باید توسط انسان کنترل شود.

In [ ]:
import csv
import json
from collections import Counter

assert emotion_metadata_path.is_file(), "Emotion metadata was not created; rerun the inference cell."
with emotion_metadata_path.open("r", encoding="utf-8", newline="") as csv_file:
    emotion_manifest = list(csv.DictReader(csv_file))
with predictions_path.open("r", encoding="utf-8") as json_file:
    saved_predictions = json.load(json_file)

assert emotion_manifest, "The emotion manifest is empty."
assert len(emotion_manifest) == len(emotion_input_rows), "Emotion row count mismatch."
predictions_by_id = {item["utterance_id"]: item for item in saved_predictions}
emotion_validation_errors = []

for row in emotion_manifest:
    try:
        utterance_id = row["utterance_id"]
        assert row["pseudo_emotion"] in CANONICAL_LABELS, "Unknown canonical label."
        confidence = float(row["pseudo_confidence"])
        margin = float(row["pseudo_margin"])
        assert 0.0 <= confidence <= 1.0, "Confidence is outside [0, 1]."
        assert 0.0 <= margin <= 1.0, "Margin is outside [0, 1]."
        prediction = predictions_by_id[utterance_id]
        probabilities = prediction["probabilities"]
        assert set(probabilities) == set(CANONICAL_LABELS), "Probability labels are incomplete."
        assert abs(sum(probabilities.values()) - 1.0) < 1e-3, "Probabilities do not sum to one."
        assert prediction["pseudo_emotion"] == row["pseudo_emotion"]
    except Exception as exc:
        emotion_validation_errors.append(f"{row.get('utterance_id', '<unknown>')}: {exc}")

if emotion_validation_errors:
    raise ValueError("Emotion validation failed:\n" + "\n".join(emotion_validation_errors[:20]))

print("Validated pseudo-labeled utterances:", len(emotion_manifest))
print("Label distribution:", dict(Counter(row["pseudo_emotion"] for row in emotion_manifest)))
print("Review queue rows:", len(review_rows))
print("Pseudo-label validation: PASSED")
print("Next: inspect low-confidence samples before using labels for training.")

## اقدام لازم در Colab برای گام ۱.۵

1. اگر خروجی گام ۱.۴ وجود ندارد، ابتدا سلول‌های ۱۰ و ۱۱ را اجرا کن.
2. سلول‌های ۱۵ تا ۱۷ را اجرا کن.
3. خروجی سلول ۱۷ را ارسال کن.
4. چند نمونه از `emotion_review.csv` را با صوت اصلی گوش بده و صحت برچسب‌ها را بررسی کن؛ این کار برای اعتماد به pseudo-labelها لازم است.

برای پردازش کامل manifest، مقدار `PSEUDO_LABEL_LIMIT` را در سلول ۱۵ به `None` تغییر بده و سلول‌های ۱۶ و ۱۷ را دوباره اجرا کن.

# فاز ۲ — Context & Emotion Extractor
## گام ۲.۱ — انتخاب مدل زبانی سبک فارسی

برای encoder متن، **ALBERT-Persian** با شناسه `HooshvareLab/albert-fa-zwnj-base-v2` انتخاب شده است. این مدل عضو خانواده مدل‌های Lite BERT فارسی است و برای استخراج representation از متن، سبک‌تر از استفاده از یک LLM چندمیلیاردپارامتری است.

نقش این مدل در پروژه:

```text
Persian text → tokenizer → ALBERT encoder → mean-pooled context vector
```

در این گام فقط encoder بارگذاری و تست می‌شود؛ **فین‌تیون و تولید Emotion/Prosody Vector نهایی در گام ۲.۲ انجام می‌شود.**

### سلول ۱۸: نصب وابستگی tokenizer و بارگذاری encoder

اگر `sentencepiece` در runtime موجود نباشد، این سلول آن را نصب می‌کند. سپس tokenizer و مدل را از Hugging Face بارگذاری و مشخصات مدل را چاپ می‌کند.

In [ ]:
import importlib.util
import json
import subprocess
import sys
from pathlib import Path

import torch
from transformers import AutoModel, AutoTokenizer

if importlib.util.find_spec("sentencepiece") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "sentencepiece"],
        check=True,
    )

PROJECT_ROOT = Path("/content/drive/MyDrive/Persian_Emotional_TTS")
TEXT_MODEL_DIR = PROJECT_ROOT / "models" / "emotion_extractor"
TEXT_MODEL_DIR.mkdir(parents=True, exist_ok=True)

TEXT_MODEL_NAME = "HooshvareLab/albert-fa-zwnj-base-v2"
MAX_TEXT_LENGTH = 128
TEXT_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading Persian text encoder:", TEXT_MODEL_NAME)
text_tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
text_model = AutoModel.from_pretrained(TEXT_MODEL_NAME).to(TEXT_DEVICE).eval()
text_model_config = text_model.config
text_parameter_count = sum(parameter.numel() for parameter in text_model.parameters())

print("Text device:", TEXT_DEVICE)
print("Model type:", text_model_config.model_type)
print("Hidden size:", text_model_config.hidden_size)
print("Trainable parameters at this stage:", text_parameter_count)
print("Maximum text length:", MAX_TEXT_LENGTH)
print("Persian text encoder: READY")

### سلول ۱۹: تست tokenization و استخراج context vector

این سلول دو جمله فارسی را به بردار تبدیل می‌کند و شکل خروجی و finite بودن مقادیر را بررسی می‌کند. بردارها فقط Smoke Test هستند و هنوز بردار احساس آموزش‌دیده محسوب نمی‌شوند.

In [ ]:
import json
import torch


def mean_pool_text(hidden_states, attention_mask):
    """Mean-pool token states while excluding padding tokens."""
    mask = attention_mask.unsqueeze(-1).to(hidden_states.dtype)
    summed = (hidden_states * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts


def encode_persian_texts(texts):
    """Encode a list of Persian strings into CPU context vectors."""
    encoded = text_tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_TEXT_LENGTH,
        return_tensors="pt",
    )
    encoded = {key: value.to(TEXT_DEVICE) for key, value in encoded.items()}
    with torch.inference_mode():
        model_output = text_model(**encoded)
        embeddings = mean_pool_text(
            model_output.last_hidden_state, encoded["attention_mask"]
        )
    return embeddings.detach().cpu(), encoded


sample_texts = [
    "دخترک با شنیدن صدای رعد، مادرش را در آغوش گرفت.",
    "امروز هوا بسیار دلپذیر و آفتابی است.",
]
sample_embeddings, sample_encoded = encode_persian_texts(sample_texts)

assert sample_embeddings.shape == (len(sample_texts), text_model_config.hidden_size)
assert torch.isfinite(sample_embeddings).all(), "Non-finite values in text embeddings."

print("Sample tokens:", text_tokenizer.convert_ids_to_tokens(sample_encoded["input_ids"][0].detach().cpu().tolist()))
print("Embedding shape:", tuple(sample_embeddings.shape))
print("First embedding norm:", round(sample_embeddings[0].norm().item(), 4))
print("Text embedding smoke test: PASSED")

text_encoder_config = {
    "model_name": TEXT_MODEL_NAME,
    "model_role": "persian_context_encoder",
    "pooling": "attention_mask_mean",
    "max_text_length": MAX_TEXT_LENGTH,
    "hidden_size": text_model_config.hidden_size,
    "parameter_count": text_parameter_count,
    "fine_tuned_for_emotion": False,
    "next_step": "2.2",
}
text_encoder_config_path = TEXT_MODEL_DIR / "text_encoder_config.json"
text_encoder_config_path.write_text(
    json.dumps(text_encoder_config, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Text encoder config:", text_encoder_config_path)

## گام ۲.۱ — encoder متن انتخاب و تست شد

مدل و tokenizer آماده‌اند و تابع `encode_persian_texts` برای استفاده در گام بعد در runtime وجود دارد. فایل `models/emotion_extractor/text_encoder_config.json` نیز مشخصات انتخاب را ذخیره می‌کند.

### گام ۲.۲ — مرحله بعد
اتصال این encoder به داده‌های متنی و برچسب‌های احساس، طراحی head برای Emotion/Prosody Vector و آغاز فین‌تیون.

## گام ۲.۲ — فین‌تیون Emotion/Prosody Encoder

در این گام encoder فارسی گام ۲.۱ با دو head آموزش می‌بیند:

1. **Emotion head:** پیش‌بینی یکی از شش برچسب احساس از روی متن.
2. **Prosody head:** پیش‌بینی بردار شش‌بعدی شامل مدت، میانگین و پراکندگی زیر (F0)، انرژی و سرعت گفتار.

برچسب احساس از pseudo-labelهای گام ۱.۵ می‌آید. targetهای prosody نیز از صوت استخراج می‌شوند و فقط **weak acoustic targets** هستند؛ بنابراین این مدل یک baseline قابل آزمایش است، نه مدل نهایی و دقیق لحن.

برای اجرای واقعی آموزش، ابتدا سلول‌های ۱۵ تا ۱۷ باید `metadata_emotion.csv` را ساخته باشند. پیش‌فرض این گام یک Smoke Test کوچک است، نه آموزش نهایی روی کل داده.

### سلول ۲۰: ساخت manifest آموزش و targetهای prosody

این سلول متن و pseudo-labelها را می‌خواند، ویژگی‌های تقریبی صوت را استخراج می‌کند و آمار نرمال‌سازی را فقط از بخش train محاسبه می‌کند.

- `TRAINING_LIMIT = 32`: محدودیت تست اولیه؛ برای کل داده `None` بگذار.
- `EXCLUDE_REVIEW = False`: برای جلوگیری از خالی‌شدن subset آزمایشی همه pseudo-labelها را نگه می‌دارد. برای آموزش جدی، بعد از بازبینی انسانی آن را `True` کن.
- صوت‌های اصلی تغییر نمی‌کنند.

In [ ]:
import csv
import json
import math
import random
from pathlib import Path

import librosa
import numpy as np

EMOTION_METADATA_PATH = PROCESSED_ROOT / "emotion" / "metadata_emotion.csv"
EMOTION_TRAINING_ROOT = PROCESSED_ROOT / "emotion_training"
EMOTION_TRAINING_ROOT.mkdir(parents=True, exist_ok=True)

EMOTION_LABELS = ("neutral", "happy", "sad", "angry", "fearful", "surprised")
EMOTION_TO_ID = {label: index for index, label in enumerate(EMOTION_LABELS)}
PROSODY_FEATURE_NAMES = (
    "log_duration", "f0_mean_hz", "f0_std_hz",
    "rms_mean", "rms_std", "words_per_second",
)
TRAINING_LIMIT = 32  # Smoke test; set to None for all pseudo-labeled rows.
EXCLUDE_REVIEW = False  # Set True after manually reviewing pseudo-labels.
RANDOM_SEED = 42

assert EMOTION_METADATA_PATH.is_file(), "Run the emotion pseudo-labeling cells first."
with EMOTION_METADATA_PATH.open("r", encoding="utf-8", newline="") as csv_file:
    emotion_source_rows = list(csv.DictReader(csv_file))

candidate_rows = []
for row in emotion_source_rows:
    if row.get("pseudo_emotion") not in EMOTION_TO_ID:
        raise ValueError(f"Unknown emotion label in {row.get('utterance_id')}")
    if EXCLUDE_REVIEW and str(row.get("emotion_review_required", "false")).lower() == "true":
        continue
    candidate_rows.append(dict(row))

assert candidate_rows, "No usable pseudo-labeled rows remain after filtering."
rng = random.Random(RANDOM_SEED)
rng.shuffle(candidate_rows)
if TRAINING_LIMIT is not None:
    candidate_rows = candidate_rows[:TRAINING_LIMIT]
assert len(candidate_rows) >= 2, "At least two rows are needed for a training smoke test."


def extract_weak_prosody_target(row):
    """Extract approximate acoustic targets; these are not human prosody labels."""
    audio, sample_rate = librosa.load(
        str(PROJECT_ROOT / row["audio_path"]), sr=22050, mono=True
    )
    if len(audio) == 0:
        raise ValueError(f"Empty audio for {row['utterance_id']}")
    duration = len(audio) / sample_rate
    text = row.get("normalized_text") or row["text"]
    word_count = max(len(text.split()), 1)

    rms = librosa.feature.rms(y=audio, frame_length=2048, hop_length=512)[0]
    f0 = librosa.yin(
        audio, fmin=50, fmax=500, sr=sample_rate,
        frame_length=2048, hop_length=512,
    )
    voiced_f0 = f0[np.isfinite(f0) & (f0 > 0)]
    f0_mean = float(voiced_f0.mean()) if len(voiced_f0) else 0.0
    f0_std = float(voiced_f0.std()) if len(voiced_f0) else 0.0

    return [
        math.log1p(duration),
        f0_mean,
        f0_std,
        float(rms.mean()),
        float(rms.std()),
        word_count / max(duration, 1e-6),
    ]


enriched_rows = []
for index, row in enumerate(candidate_rows):
    row["finetune_text"] = row.get("normalized_text") or row["text"]
    row["prosody_target"] = extract_weak_prosody_target(row)
    row["sample_weight"] = max(float(row.get("pseudo_confidence", 1.0)), 0.1)
    enriched_rows.append(row)

validation_count = int(round(len(enriched_rows) * 0.1)) if len(enriched_rows) >= 10 else 0
validation_rows = enriched_rows[:validation_count]
train_rows = enriched_rows[validation_count:]
assert train_rows, "The training split is empty."

train_target_array = np.asarray(
    [row["prosody_target"] for row in train_rows], dtype=np.float32
)
PROSODY_MEAN = train_target_array.mean(axis=0)
PROSODY_STD = np.maximum(train_target_array.std(axis=0), 1e-6)

for row in train_rows + validation_rows:
    target = np.asarray(row["prosody_target"], dtype=np.float32)
    row["prosody_target_normalized"] = ((target - PROSODY_MEAN) / PROSODY_STD).tolist()

training_manifest = {
    "emotion_labels": list(EMOTION_LABELS),
    "prosody_feature_names": list(PROSODY_FEATURE_NAMES),
    "prosody_target_source": "weak_acoustic_features",
    "prosody_mean": PROSODY_MEAN.tolist(),
    "prosody_std": PROSODY_STD.tolist(),
    "train": train_rows,
    "validation": validation_rows,
}
training_manifest_path = EMOTION_TRAINING_ROOT / "text_training_manifest.json"
training_manifest_path.write_text(
    json.dumps(training_manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

print("Training rows:", len(train_rows))
print("Validation rows:", len(validation_rows))
print("Prosody features:", PROSODY_FEATURE_NAMES)
print("Training manifest:", training_manifest_path)

### سلول ۲۱: تعریف مدل چندوظیفه‌ای و DataLoader

مدل از encoder گام ۲.۱ استفاده می‌کند و دو head جدا دارد. خروجی `prosody_vector` در فضای نرمال‌شده targetها آموزش می‌بیند و هنگام inference با mean/std ذخیره‌شده به واحد اصلی برگردانده می‌شود.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

BATCH_SIZE = 4


def masked_mean_pool(hidden_states, attention_mask):
    mask = attention_mask.unsqueeze(-1).to(hidden_states.dtype)
    return (hidden_states * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)


class EmotionProsodyEncoder(nn.Module):
    def __init__(self, encoder, hidden_size, emotion_count, prosody_dim):
        super().__init__()
        self.encoder = encoder
        self.dropout = nn.Dropout(0.1)
        self.emotion_head = nn.Linear(hidden_size, emotion_count)
        self.prosody_head = nn.Linear(hidden_size, prosody_dim)

    def forward(self, input_ids, attention_mask, **encoder_kwargs):
        encoder_output = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            **encoder_kwargs,
        )
        context = masked_mean_pool(encoder_output.last_hidden_state, attention_mask)
        context = self.dropout(context)
        return {
            "context": context,
            "emotion_logits": self.emotion_head(context),
            "prosody_vector": self.prosody_head(context),
        }


class TextEmotionProsodyDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        return {
            "text": row["finetune_text"],
            "emotion_id": EMOTION_TO_ID[row["pseudo_emotion"]],
            "prosody_target": row["prosody_target_normalized"],
            "sample_weight": row["sample_weight"],
        }


def collate_emotion_prosody(batch):
    encoded = text_tokenizer(
        [item["text"] for item in batch],
        padding=True,
        truncation=True,
        max_length=MAX_TEXT_LENGTH,
        return_tensors="pt",
    )
    encoded.update({
        "emotion_labels": torch.tensor(
            [item["emotion_id"] for item in batch], dtype=torch.long
        ),
        "prosody_targets": torch.tensor(
            [item["prosody_target"] for item in batch], dtype=torch.float32
        ),
        "sample_weight": torch.tensor(
            [item["sample_weight"] for item in batch], dtype=torch.float32
        ),
    })
    return encoded


train_dataset = TextEmotionProsodyDataset(train_rows)
validation_dataset = TextEmotionProsodyDataset(validation_rows)
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_emotion_prosody, num_workers=0,
)
validation_loader = (
    DataLoader(
        validation_dataset, batch_size=BATCH_SIZE, shuffle=False,
        collate_fn=collate_emotion_prosody, num_workers=0,
    )
    if validation_rows else None
)

multi_task_model = EmotionProsodyEncoder(
    encoder=text_model,
    hidden_size=text_model_config.hidden_size,
    emotion_count=len(EMOTION_LABELS),
    prosody_dim=len(PROSODY_FEATURE_NAMES),
).to(TEXT_DEVICE)

print("Model parameters:", sum(p.numel() for p in multi_task_model.parameters()))
print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader) if validation_loader else 0)
print("Multi-task model: READY")

### سلول ۲۲: اجرای فین‌تیون Smoke Test

`TRAINING_EPOCHS = 1` فقط برای بررسی سلامت حلقه آموزش است. برای آموزش واقعی باید داده بیشتر و بازبینی‌شده، split مناسب، تنظیم epoch و ارزیابی جداگانه داشته باشیم. confidence هر pseudo-label به‌عنوان وزن نمونه در loss استفاده می‌شود.

In [ ]:
import torch
import torch.nn.functional as F

TRAINING_EPOCHS = 1
LEARNING_RATE = 2e-5
PROSODY_LOSS_WEIGHT = 0.2
WEIGHT_DECAY = 0.01

optimizer = torch.optim.AdamW(
    multi_task_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)


def move_batch_to_text_device(batch):
    return {key: value.to(TEXT_DEVICE) for key, value in batch.items()}


def run_emotion_prosody_epoch(loader, training):
    if loader is None:
        return None
    multi_task_model.train(training)
    totals = {"loss": 0.0, "emotion": 0.0, "prosody": 0.0, "correct": 0, "count": 0}

    with torch.set_grad_enabled(training):
        for batch in loader:
            batch = move_batch_to_text_device(batch)
            if training:
                optimizer.zero_grad(set_to_none=True)

            outputs = multi_task_model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                **{
                    key: value for key, value in batch.items()
                    if key in {"token_type_ids", "position_ids"}
                },
            )
            emotion_per_sample = F.cross_entropy(
                outputs["emotion_logits"], batch["emotion_labels"], reduction="none"
            )
            prosody_per_sample = F.mse_loss(
                outputs["prosody_vector"], batch["prosody_targets"], reduction="none"
            ).mean(dim=1)
            weights = batch["sample_weight"]
            normalizer = weights.sum().clamp(min=1e-8)
            emotion_loss = (emotion_per_sample * weights).sum() / normalizer
            prosody_loss = (prosody_per_sample * weights).sum() / normalizer
            loss = emotion_loss + PROSODY_LOSS_WEIGHT * prosody_loss

            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(multi_task_model.parameters(), max_norm=1.0)
                optimizer.step()

            batch_size = batch["emotion_labels"].shape[0]
            totals["loss"] += float(loss.detach()) * batch_size
            totals["emotion"] += float(emotion_loss.detach()) * batch_size
            totals["prosody"] += float(prosody_loss.detach()) * batch_size
            totals["correct"] += int(
                (outputs["emotion_logits"].argmax(dim=-1) == batch["emotion_labels"])
                .sum().detach().cpu()
            )
            totals["count"] += batch_size

    count = max(totals["count"], 1)
    return {
        "loss": totals["loss"] / count,
        "emotion_loss": totals["emotion"] / count,
        "prosody_loss": totals["prosody"] / count,
        "emotion_accuracy": totals["correct"] / count,
    }


training_history = []
for epoch in range(1, TRAINING_EPOCHS + 1):
    train_metrics = run_emotion_prosody_epoch(train_loader, training=True)
    validation_metrics = run_emotion_prosody_epoch(validation_loader, training=False)
    history_entry = {"epoch": epoch, "train": train_metrics, "validation": validation_metrics}
    training_history.append(history_entry)
    print("Epoch:", epoch, "| train:", train_metrics, "| validation:", validation_metrics)

print("Fine-tuning loop: COMPLETE")

### سلول ۲۳: ذخیره checkpoint و تست استخراج بردار

checkpoint شامل وزن encoder و headها، نام کلاس‌ها، mean/std مربوط به prosody و تنظیمات آموزش است. در تست نهایی، خروجی احساس به احتمال و خروجی prosody از فضای نرمال‌شده به واحد تقریبی اصلی برگردانده می‌شود.

In [ ]:
import json
import torch

checkpoint_path = TEXT_MODEL_DIR / "emotion_prosody_encoder.pt"
checkpoint = {
    "encoder_model_name": TEXT_MODEL_NAME,
    "model_state_dict": {
        key: value.detach().cpu()
        for key, value in multi_task_model.state_dict().items()
    },
    "emotion_labels": list(EMOTION_LABELS),
    "prosody_feature_names": list(PROSODY_FEATURE_NAMES),
    "prosody_mean": PROSODY_MEAN.tolist(),
    "prosody_std": PROSODY_STD.tolist(),
    "max_text_length": MAX_TEXT_LENGTH,
    "label_source": "pseudo",
    "prosody_target_source": "weak_acoustic_features",
    "training_config": {
        "limit": TRAINING_LIMIT,
        "epochs": TRAINING_EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "prosody_loss_weight": PROSODY_LOSS_WEIGHT,
        "exclude_review": EXCLUDE_REVIEW,
    },
    "history": training_history,
}
torch.save(checkpoint, checkpoint_path)

multi_task_model.eval()
inference_texts = [
    "او با ترس به دربارهٔ تاریک نگاه کرد.",
    "بچه‌ها با شادی در باغ بازی می‌کردند.",
]
inference_encoded = text_tokenizer(
    inference_texts,
    padding=True,
    truncation=True,
    max_length=MAX_TEXT_LENGTH,
    return_tensors="pt",
)
inference_encoded = {key: value.to(TEXT_DEVICE) for key, value in inference_encoded.items()}
with torch.inference_mode():
    inference_output = multi_task_model(**inference_encoded)
    emotion_probabilities = torch.softmax(inference_output["emotion_logits"], dim=-1)
    normalized_prosody = inference_output["prosody_vector"]

prosody_mean_tensor = torch.tensor(
    PROSODY_MEAN, dtype=normalized_prosody.dtype, device=normalized_prosody.device
)
prosody_std_tensor = torch.tensor(
    PROSODY_STD, dtype=normalized_prosody.dtype, device=normalized_prosody.device
)
prosody_vectors = normalized_prosody * prosody_std_tensor + prosody_mean_tensor

sample_vectors = []
for index, text in enumerate(inference_texts):
    probabilities = emotion_probabilities[index].detach().cpu()
    predicted_id = int(probabilities.argmax())
    sample_vectors.append({
        "text": text,
        "predicted_emotion": EMOTION_LABELS[predicted_id],
        "confidence": float(probabilities[predicted_id]),
        "emotion_probabilities": {
            label: float(probabilities[label_index])
            for label_index, label in enumerate(EMOTION_LABELS)
        },
        "prosody_vector": prosody_vectors[index].detach().cpu().tolist(),
    })

sample_vectors_path = TEXT_MODEL_DIR / "sample_emotion_prosody_vectors.json"
sample_vectors_path.write_text(
    json.dumps(sample_vectors, ensure_ascii=False, indent=2), encoding="utf-8"
)

assert checkpoint_path.is_file() and checkpoint_path.stat().st_size > 0
assert torch.isfinite(prosody_vectors).all()
print("Checkpoint:", checkpoint_path)
print("Sample vectors:", sample_vectors_path)
print("Emotion/prosody inference smoke test: PASSED")

## گام ۲.۲ — baseline فین‌تیون اضافه شد

اکنون مسیر زیر در نوت‌بوک موجود است:

```text
Persian text → ALBERT encoder → emotion probabilities + prosody vector
```

این آموزش با pseudo-label و targetهای تقریبی صوتی انجام می‌شود و برای تصمیم نهایی درباره کیفیت، باید روی داده بیشتر و برچسب‌های بازبینی‌شده ارزیابی شود. فایل checkpoint برای مراحل بعد در `models/emotion_extractor/emotion_prosody_encoder.pt` ذخیره می‌شود.

گام بعدی roadmap: **فاز ۳، گام ۳.۱ — پیاده‌سازی مدل آکوستیک Flow Matching مشابه Matcha-TTS**.

# فاز ۳ — Acoustic Model & Vocoder
## گام ۳.۱ — پیاده‌سازی Flow Matching مشابه Matcha-TTS

در این گام یک مدل آکوستیک compact بر پایه **Conditional Flow Matching (CFM)** پیاده‌سازی می‌کنیم. مدل یک vector field یاد می‌گیرد که نویز را به mel-spectrogram هدایت کند و هنگام inference با چند گام Euler صوتِ ویژگی‌محور تولید می‌شود.

این پیاده‌سازی **Matcha-inspired** است و برای اتصال به encoder فارسی و شرط احساس آماده می‌شود؛ مدل رسمی Matcha-TTS یا وزن‌های ازپیش‌آموزش‌دیده آن کپی نمی‌شود.

ورودی و خروجی:

```text
condition: [batch, frames, condition_dim]
mel:       [batch, 80, frames]
```

در این گام فقط معماری، CFM loss و sampler تست می‌شوند. آموزش روی داده فارسی و اتصال بردار احساس در گام ۳.۲ انجام می‌شود.

### سلول ۲۴: تعریف vector field و بلوک‌های conditioning

مدل از time embedding، projection شرط و بلوک‌های residual با FiLM استفاده می‌کند. تنظیمات mel نیز برای اتصال بعدی به صوت ۲۲۰۵۰ هرتز ذخیره می‌شود.

In [ ]:
import json
import math
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

ACOUSTIC_MODEL_DIR = PROJECT_ROOT / "models" / "acoustic"
ACOUSTIC_MODEL_DIR.mkdir(parents=True, exist_ok=True)

MEL_CONFIG = {
    "sample_rate": 22050,
    "n_fft": 1024,
    "win_length": 1024,
    "hop_length": 256,
    "n_mels": 80,
    "f_min": 0.0,
    "f_max": 8000.0,
}
FLOW_CONFIG = {
    "mel_channels": MEL_CONFIG["n_mels"],
    "condition_dim": 128,
    "hidden_channels": 128,
    "time_dim": 64,
    "num_blocks": 4,
    "sample_steps": 10,
}


class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dimension):
        super().__init__()
        self.dimension = dimension

    def forward(self, time):
        half = self.dimension // 2
        frequencies = torch.exp(
            torch.arange(half, device=time.device, dtype=torch.float32)
            * (-math.log(10000.0) / max(half - 1, 1))
        )
        angles = time.float().unsqueeze(1) * frequencies.unsqueeze(0) * 1000.0
        embedding = torch.cat([angles.sin(), angles.cos()], dim=-1)
        if self.dimension % 2:
            embedding = F.pad(embedding, (0, 1))
        return embedding


class FiLMResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, channels)
        self.norm2 = nn.GroupNorm(8, channels)
        self.conv1 = nn.Conv1d(channels, channels, 3, padding=1)
        self.conv2 = nn.Conv1d(channels, channels, 3, padding=1)
        self.film = nn.Linear(channels, channels * 2)

    def forward(self, inputs, conditioning):
        hidden = self.conv1(F.silu(self.norm1(inputs)))
        scale, shift = self.film(conditioning).unsqueeze(-1).chunk(2, dim=1)
        hidden = hidden * (1.0 + scale) + shift
        hidden = self.conv2(F.silu(self.norm2(hidden)))
        return inputs + hidden


class ConditionalVectorField(nn.Module):
    def __init__(self, mel_channels, condition_dim, hidden_channels, time_dim, num_blocks):
        super().__init__()
        self.time_embedding = SinusoidalTimeEmbedding(time_dim)
        self.time_mlp = nn.Sequential(
            nn.Linear(time_dim, hidden_channels),
            nn.SiLU(),
            nn.Linear(hidden_channels, hidden_channels),
        )
        self.input_conv = nn.Conv1d(mel_channels, hidden_channels, 3, padding=1)
        self.condition_conv = nn.Conv1d(condition_dim, hidden_channels, 1)
        self.blocks = nn.ModuleList([
            FiLMResidualBlock(hidden_channels) for _ in range(num_blocks)
        ])
        self.output_norm = nn.GroupNorm(8, hidden_channels)
        self.output_conv = nn.Conv1d(hidden_channels, mel_channels, 1)
        nn.init.zeros_(self.output_conv.weight)
        nn.init.zeros_(self.output_conv.bias)

    def forward(self, noisy_mel, time, condition):
        # condition is [batch, frames, condition_dim].
        condition_hidden = self.condition_conv(condition.transpose(1, 2))
        hidden = self.input_conv(noisy_mel) + condition_hidden
        time_hidden = self.time_mlp(self.time_embedding(time))
        global_condition = condition_hidden.mean(dim=-1)
        conditioning = time_hidden + global_condition
        for block in self.blocks:
            hidden = block(hidden, conditioning)
        return self.output_conv(F.silu(self.output_norm(hidden)))

### سلول ۲۵: CFM loss، Euler sampler و Smoke Test

هدف آموزش، پیش‌بینی velocity بین نویز `x0` و mel واقعی `x1` است:

```text
x_t = (1 - t) * x0 + t * x1
target_velocity = x1 - x0
```

سلول همچنین mask برای فریم‌های معتبر را پشتیبانی می‌کند و با backward مشخص می‌کند که مسیر گرادیان سالم است.

In [ ]:
import json
import torch
import torch.nn.functional as F


class ConditionalFlowMatchingAcousticModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = dict(config)
        self.vector_field = ConditionalVectorField(
            mel_channels=config["mel_channels"],
            condition_dim=config["condition_dim"],
            hidden_channels=config["hidden_channels"],
            time_dim=config["time_dim"],
            num_blocks=config["num_blocks"],
        )

    def forward(self, mel, condition, mask=None):
        assert mel.ndim == 3, "mel must be [batch, mel_channels, frames]"
        assert condition.ndim == 3, "condition must be [batch, frames, condition_dim]"
        assert mel.shape[0] == condition.shape[0]
        assert mel.shape[2] == condition.shape[1]

        noise = torch.randn_like(mel)
        time = torch.rand(mel.shape[0], device=mel.device, dtype=mel.dtype)
        time_view = time[:, None, None]
        noisy_mel = (1.0 - time_view) * noise + time_view * mel
        target_velocity = mel - noise
        predicted_velocity = self.vector_field(noisy_mel, time, condition)
        squared_error = (predicted_velocity - target_velocity).square()

        if mask is not None:
            valid = mask.to(squared_error.dtype).unsqueeze(1)
            loss = (squared_error * valid).sum() / (
                valid.sum() * mel.shape[1]
            ).clamp(min=1.0)
        else:
            loss = squared_error.mean()
        return {
            "loss": loss,
            "predicted_velocity": predicted_velocity,
            "target_velocity": target_velocity,
            "time": time,
        }

    @torch.no_grad()
    def sample(self, condition, steps=None, mask=None):
        steps = steps or self.config["sample_steps"]
        assert steps >= 1
        batch_size, frames, _ = condition.shape
        device = condition.device
        dtype = self.vector_field.input_conv.weight.dtype
        sample = torch.randn(
            batch_size, self.config["mel_channels"], frames,
            device=device, dtype=dtype,
        )
        step_size = 1.0 / steps
        for step in range(steps):
            time = torch.full(
                (batch_size,), step / steps, device=device, dtype=dtype
            )
            velocity = self.vector_field(sample, time, condition)
            sample = sample + velocity * step_size
        if mask is not None:
            sample = sample * mask.to(sample.dtype).unsqueeze(1)
        return sample


ACOUSTIC_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
flow_model = ConditionalFlowMatchingAcousticModel(FLOW_CONFIG).to(ACOUSTIC_DEVICE)

smoke_condition = torch.randn(2, 24, FLOW_CONFIG["condition_dim"], device=ACOUSTIC_DEVICE)
smoke_mel = torch.randn(2, FLOW_CONFIG["mel_channels"], 24, device=ACOUSTIC_DEVICE)
smoke_mask = torch.tensor(
    [[1] * 18 + [0] * 6, [1] * 24], device=ACOUSTIC_DEVICE, dtype=torch.float32
)
smoke_output = flow_model(smoke_mel, smoke_condition, mask=smoke_mask)
smoke_output["loss"].backward()
assert any(parameter.grad is not None for parameter in flow_model.parameters())
generated_mel = flow_model.sample(smoke_condition, steps=4, mask=smoke_mask)
assert generated_mel.shape == smoke_mel.shape
assert torch.isfinite(generated_mel).all()

flow_config_path = ACOUSTIC_MODEL_DIR / "flow_matching_config.json"
flow_config_path.write_text(
    json.dumps({
        "architecture": "compact_matcha_inspired_cfm",
        "mel": MEL_CONFIG,
        "flow": FLOW_CONFIG,
        "condition_layout": "[batch, frames, condition_dim]",
        "mel_layout": "[batch, mel_channels, frames]",
        "objective": "target_velocity = mel - noise",
        "sampler": "euler",
        "training_status": "architecture_smoke_tested_not_trained",
    }, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("CFM loss:", round(float(smoke_output["loss"].detach()), 6))
print("Generated mel shape:", tuple(generated_mel.shape))
print("Flow Matching architecture smoke test: PASSED")
print("Architecture config:", flow_config_path)

## گام ۳.۱ — معماری آکوستیک آماده شد

کلاس `ConditionalFlowMatchingAcousticModel` در runtime موجود است و مسیر آموزش و sampling آن تست شده است. این مدل هنوز روی داده فارسی آموزش ندیده است.

در **گام ۳.۲** باید melهای واقعی، conditioning متن و بردارهای احساس را به همین مدل وصل کنیم و حلقه آموزش روی داده فارسی را اجرا کنیم.

## گام ۳.۲ — آموزش مدل آکوستیک با شرطی‌سازی احساس

در این گام مدل CFM گام ۳.۱ با mel واقعی صوتهای فارسی آموزش می‌بیند. برای هر نمونه، conditioning از سه جزء ساخته می‌شود:

```text
ALBERT context + emotion probabilities + prosody vector
```

یک projection آموزش‌پذیر این بردار را به `condition_dim = 128` مدل تبدیل می‌کند. در این baseline، conditioning سراسری در تمام فریم‌های mel تکرار می‌شود؛ alignment در سطح واج و duration modeling هنوز بخش توسعه بعدی است.

برای اجرای این گام باید خروجی گام‌های ۱.۵، ۲.۱، ۲.۲ و ۳.۱ در runtime یا Drive موجود باشد. پیش‌فرض ۴ نمونه و ۱ epoch فقط برای Smoke Test است.

### سلول ۲۶: ساخت mel و condition cache

melها با sample rate و تنظیمات گام ۳.۱ و با قرارداد log-mel مورد انتظار BigVGAN ساخته و در Drive cache می‌شوند. STFT با `center=False` و log-clamp انجام می‌شود تا ورودی آموزش و ووکودر یکسان باشد. conditioning نیز با encoder فین‌تیون‌شده گام ۲.۲ و بدون گرادیان ساخته می‌شود.

`MAX_MEL_FRAMES = 512` برای محدودکردن حافظه در تست اولیه است. برای داده واقعی و جمله‌های طولانی باید strategy برش یا مقدار بزرگ‌تر انتخاب شود.

In [ ]:
import csv
import json
import random
from pathlib import Path

import librosa
import numpy as np
import torch
import torch.nn.functional as F

ACOUSTIC_TRAINING_LIMIT = 4  # Smoke test; set to None for all available rows.
ACOUSTIC_EXCLUDE_REVIEW = False
MAX_MEL_FRAMES = 512  # Increase or add chunking for longer utterances.
ACOUSTIC_RANDOM_SEED = 42
MEL_CONVENTION = "bigvgan_log_mel_v1"
MEL_CACHE_DIR = PROCESSED_ROOT / "mel_cache_bigvgan_v1"
CONDITION_CACHE_DIR = PROCESSED_ROOT / "condition_cache" / "emotion_prosody_v1"
MEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
CONDITION_CACHE_DIR.mkdir(parents=True, exist_ok=True)

assert "multi_task_model" in globals(), "Run the step 2.2 fine-tuning cells first."
assert "flow_model" in globals(), "Run the step 3.1 architecture cells first."
with EMOTION_METADATA_PATH.open("r", encoding="utf-8", newline="") as csv_file:
    acoustic_source_rows = list(csv.DictReader(csv_file))

acoustic_rows = []
for row in acoustic_source_rows:
    if ACOUSTIC_EXCLUDE_REVIEW and str(row.get("emotion_review_required", "false")).lower() == "true":
        continue
    acoustic_rows.append(dict(row))
assert acoustic_rows, "No acoustic training rows are available."
random.Random(ACOUSTIC_RANDOM_SEED).shuffle(acoustic_rows)
if ACOUSTIC_TRAINING_LIMIT is not None:
    acoustic_rows = acoustic_rows[:ACOUSTIC_TRAINING_LIMIT]
assert acoustic_rows, "The selected acoustic subset is empty."


def mel_cache_path(row):
    suffix = "full" if MAX_MEL_FRAMES is None else str(MAX_MEL_FRAMES)
    return MEL_CACHE_DIR / f"{row['utterance_id']}_{suffix}.npy"


def condition_cache_path(row):
    return CONDITION_CACHE_DIR / f"{row['utterance_id']}.npy"


def bigvgan_log_mel(waveform, config):
    """Compute log-mel features using NVIDIA BigVGAN's STFT convention."""
    mel_basis = torch.from_numpy(librosa.filters.mel(
        sr=config["sample_rate"],
        n_fft=config["n_fft"],
        n_mels=config["n_mels"],
        fmin=config["f_min"],
        fmax=config["f_max"],
    )).float()
    window = torch.hann_window(config["win_length"])
    padding = (config["n_fft"] - config["hop_length"]) // 2
    waveform = F.pad(
        waveform.unsqueeze(1), (padding, padding), mode="reflect"
    ).squeeze(1)
    spectrum = torch.stft(
        waveform,
        n_fft=config["n_fft"],
        hop_length=config["hop_length"],
        win_length=config["win_length"],
        window=window,
        center=False,
        pad_mode="reflect",
        normalized=False,
        onesided=True,
        return_complex=True,
    )
    magnitude = torch.sqrt(torch.view_as_real(spectrum).pow(2).sum(-1) + 1e-9)
    mel_magnitude = torch.matmul(mel_basis, magnitude)
    return torch.log(torch.clamp(mel_magnitude, min=1e-5))


def ensure_mel_cache(row):
    cache_path = mel_cache_path(row)
    if cache_path.exists():
        return cache_path
    audio, sample_rate = librosa.load(
        str(PROJECT_ROOT / row["audio_path"]),
        sr=MEL_CONFIG["sample_rate"], mono=True,
    )
    if len(audio) == 0:
        raise ValueError(f"Empty audio for {row['utterance_id']}")
    waveform = torch.from_numpy(audio).float().unsqueeze(0)
    mel_log = bigvgan_log_mel(waveform, MEL_CONFIG).squeeze(0).numpy().astype(np.float32)
    if MAX_MEL_FRAMES is not None:
        mel_log = mel_log[:, :MAX_MEL_FRAMES]
    np.save(cache_path, mel_log)
    return cache_path


def ensure_condition_cache(row):
    cache_path = condition_cache_path(row)
    if cache_path.exists():
        return cache_path
    text = row.get("normalized_text") or row["text"]
    encoded = text_tokenizer(
        [text], padding=True, truncation=True,
        max_length=MAX_TEXT_LENGTH, return_tensors="pt",
    )
    encoded = {key: value.to(TEXT_DEVICE) for key, value in encoded.items()}
    multi_task_model.eval()
    with torch.inference_mode():
        output = multi_task_model(**encoded)
        emotion_probabilities = torch.softmax(output["emotion_logits"], dim=-1)
        raw_condition = torch.cat([
            output["context"], emotion_probabilities, output["prosody_vector"]
        ], dim=-1)
    np.save(cache_path, raw_condition.squeeze(0).detach().cpu().numpy().astype(np.float32))
    return cache_path


for row in acoustic_rows:
    ensure_mel_cache(row)
    ensure_condition_cache(row)

validation_count = 1 if len(acoustic_rows) >= 4 else 0
acoustic_validation_rows = acoustic_rows[:validation_count]
acoustic_train_rows = acoustic_rows[validation_count:]
assert acoustic_train_rows, "Acoustic training split is empty."
RAW_CONDITION_DIM = int(np.load(condition_cache_path(acoustic_rows[0])).shape[0])
assert RAW_CONDITION_DIM > 0

acoustic_manifest_path = PROCESSED_ROOT / "acoustic_training_manifest.json"
acoustic_manifest_path.write_text(
    json.dumps({
        "train_ids": [row["utterance_id"] for row in acoustic_train_rows],
        "validation_ids": [row["utterance_id"] for row in acoustic_validation_rows],
        "raw_condition_dim": RAW_CONDITION_DIM,
        "conditioning": "ALBERT context + emotion probabilities + normalized prosody",
        "max_mel_frames": MAX_MEL_FRAMES,
        "mel_config": MEL_CONFIG,
        "mel_convention": MEL_CONVENTION,
        "mel_normalization": "BigVGAN log-mel: log(clamp(mel_magnitude, min=1e-5)); STFT center=False",
        "limit": ACOUSTIC_TRAINING_LIMIT,
    }, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Acoustic train rows:", len(acoustic_train_rows))
print("Acoustic validation rows:", len(acoustic_validation_rows))
print("Raw condition dimension:", RAW_CONDITION_DIM)
print("Acoustic feature cache: READY")

### سلول ۲۷: Dataset، padding و projection شرط

این سلول batchهای با طول متغیر را pad می‌کند و mask می‌سازد. projection شرط همراه مدل آکوستیک آموزش می‌بیند، اما encoder متن در این گام ثابت می‌ماند.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

ACOUSTIC_BATCH_SIZE = 2


class AcousticConditionDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        mel = np.load(mel_cache_path(row)).astype(np.float32)
        condition = np.load(condition_cache_path(row)).astype(np.float32)
        return {
            "mel": torch.from_numpy(mel),
            "raw_condition": torch.from_numpy(condition),
            "length": mel.shape[1],
            "utterance_id": row["utterance_id"],
        }


def collate_acoustic_condition(batch):
    lengths = torch.tensor([item["length"] for item in batch], dtype=torch.long)
    max_frames = int(lengths.max())
    mel = torch.zeros(
        len(batch), MEL_CONFIG["n_mels"], max_frames, dtype=torch.float32
    )
    for index, item in enumerate(batch):
        mel[index, :, :item["length"]] = item["mel"]
    mask = torch.arange(max_frames)[None, :] < lengths[:, None]
    return {
        "mel": mel,
        "raw_condition": torch.stack([item["raw_condition"] for item in batch]),
        "mask": mask,
        "lengths": lengths,
        "utterance_ids": [item["utterance_id"] for item in batch],
    }


acoustic_train_dataset = AcousticConditionDataset(acoustic_train_rows)
acoustic_validation_dataset = AcousticConditionDataset(acoustic_validation_rows)
acoustic_train_loader = DataLoader(
    acoustic_train_dataset, batch_size=ACOUSTIC_BATCH_SIZE,
    shuffle=True, collate_fn=collate_acoustic_condition, num_workers=0,
)
acoustic_validation_loader = (
    DataLoader(
        acoustic_validation_dataset, batch_size=ACOUSTIC_BATCH_SIZE,
        shuffle=False, collate_fn=collate_acoustic_condition, num_workers=0,
    )
    if acoustic_validation_rows else None
)

condition_projector = nn.Sequential(
    nn.Linear(RAW_CONDITION_DIM, FLOW_CONFIG["condition_dim"]),
    nn.SiLU(),
    nn.Linear(FLOW_CONFIG["condition_dim"], FLOW_CONFIG["condition_dim"]),
).to(ACOUSTIC_DEVICE)

print("Acoustic training batches:", len(acoustic_train_loader))
print("Condition projector:", RAW_CONDITION_DIM, "->", FLOW_CONFIG["condition_dim"])
print("Acoustic DataLoader: READY")

### سلول ۲۸: آموزش CFM با conditioning احساس

در هر batch، بردار شرط به تعداد فریم‌های mel تکرار می‌شود و mask باعث می‌شود فریم‌های padding در loss بی‌تأثیر باشند. پیش‌فرض فقط ۱ epoch برای بررسی سلامت آموزش است.

In [ ]:
import torch

ACOUSTIC_EPOCHS = 1
ACOUSTIC_LEARNING_RATE = 1e-4
ACOUSTIC_WEIGHT_DECAY = 1e-4

acoustic_optimizer = torch.optim.AdamW(
    list(flow_model.parameters()) + list(condition_projector.parameters()),
    lr=ACOUSTIC_LEARNING_RATE, weight_decay=ACOUSTIC_WEIGHT_DECAY,
)


def run_acoustic_epoch(loader, training):
    if loader is None:
        return None
    flow_model.train(training)
    condition_projector.train(training)
    total_loss = 0.0
    total_batches = 0

    with torch.set_grad_enabled(training):
        for batch in loader:
            mel = batch["mel"].to(ACOUSTIC_DEVICE)
            raw_condition = batch["raw_condition"].to(ACOUSTIC_DEVICE)
            mask = batch["mask"].to(ACOUSTIC_DEVICE)
            if training:
                acoustic_optimizer.zero_grad(set_to_none=True)

            projected_condition = condition_projector(raw_condition)
            condition = projected_condition.unsqueeze(1).expand(-1, mel.shape[2], -1)
            model_output = flow_model(mel, condition, mask=mask)
            loss = model_output["loss"]

            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    list(flow_model.parameters()) + list(condition_projector.parameters()),
                    max_norm=1.0,
                )
                acoustic_optimizer.step()

            total_loss += float(loss.detach())
            total_batches += 1

    return total_loss / max(total_batches, 1)


acoustic_training_history = []
for epoch in range(1, ACOUSTIC_EPOCHS + 1):
    train_loss = run_acoustic_epoch(acoustic_train_loader, training=True)
    validation_loss = run_acoustic_epoch(acoustic_validation_loader, training=False)
    acoustic_training_history.append({
        "epoch": epoch, "train_loss": train_loss, "validation_loss": validation_loss
    })
    print("Acoustic epoch:", epoch, "| train loss:", train_loss, "| validation loss:", validation_loss)

print("Acoustic training loop: COMPLETE")

### سلول ۲۹: ذخیره مدل و تست تولید mel با شرط احساس

checkpoint شامل وزن CFM و projection شرط است. تست تولید فقط سلامت مسیر inference را بررسی می‌کند؛ کیفیت صوت پس از آموزش کافی و اتصال ووکودر در گام ۳.۳ ارزیابی می‌شود.

In [ ]:
import json
import numpy as np
import torch

acoustic_checkpoint_path = ACOUSTIC_MODEL_DIR / "flow_matching_emotion.pt"
torch.save({
    "flow_model_state_dict": {
        key: value.detach().cpu() for key, value in flow_model.state_dict().items()
    },
    "condition_projector_state_dict": {
        key: value.detach().cpu() for key, value in condition_projector.state_dict().items()
    },
    "flow_config": FLOW_CONFIG,
    "mel_config": MEL_CONFIG,
    "raw_condition_dim": RAW_CONDITION_DIM,
    "conditioning": "global_text_emotion_prosody_repeated_over_frames",
    "mel_convention": MEL_CONVENTION,
    "mel_normalization": "BigVGAN log-mel: log(clamp(mel_magnitude, min=1e-5)); STFT center=False",
    "training_config": {
        "limit": ACOUSTIC_TRAINING_LIMIT,
        "epochs": ACOUSTIC_EPOCHS,
        "batch_size": ACOUSTIC_BATCH_SIZE,
        "learning_rate": ACOUSTIC_LEARNING_RATE,
        "max_mel_frames": MAX_MEL_FRAMES,
    },
    "history": acoustic_training_history,
    "training_status": "smoke_trained_not_production",
}, acoustic_checkpoint_path)

sample_batch = next(iter(acoustic_train_loader))
sample_mel = sample_batch["mel"].to(ACOUSTIC_DEVICE)
sample_raw_condition = sample_batch["raw_condition"].to(ACOUSTIC_DEVICE)
sample_mask = sample_batch["mask"].to(ACOUSTIC_DEVICE)
flow_model.eval()
condition_projector.eval()
with torch.inference_mode():
    sample_projected = condition_projector(sample_raw_condition)
    sample_condition = sample_projected.unsqueeze(1).expand(-1, sample_mel.shape[2], -1)
    generated_mel = flow_model.sample(sample_condition, steps=4, mask=sample_mask)

assert generated_mel.shape == sample_mel.shape
assert torch.isfinite(generated_mel).all()
smoke_mel_path = ACOUSTIC_MODEL_DIR / "smoke_generated_mel.npy"
np.save(smoke_mel_path, generated_mel[0].detach().cpu().numpy())

print("Acoustic checkpoint:", acoustic_checkpoint_path)
print("Generated mel shape:", tuple(generated_mel.shape))
print("Smoke mel:", smoke_mel_path)
print("Emotion-conditioned acoustic inference: PASSED")

## گام ۳.۲ — آموزش baseline با شرط احساس اضافه شد

اکنون مسیر آموزش مدل آکوستیک با conditioning متن و احساس در نوت‌بوک موجود است. این هنوز یک baseline با conditioning سراسری و محدودیت طول است؛ برای کیفیت واقعی به داده بیشتر، آموزش طولانی‌تر و بهبود alignment نیاز داریم.

گام بعدی: **۳.۳ — اتصال BigVGAN برای تبدیل mel به صوت خام**.

## گام ۳.۳ — اتصال BigVGAN و تولید صوت خام

مدل `nvidia/bigvgan_22khz_80band` انتخاب شده است چون با تنظیمات پروژه (`22050 Hz`، ۸۰ mel band و `fmax=8000`) سازگار است.

قبل از inference، قرارداد mel گام ۳.۲ به log-mel مورد انتظار BigVGAN تغییر کرده است تا خروجی CFM با ورودی ووکودر ناسازگار نباشد. در این گام دو فایل ساخته می‌شود:

- **Reference reconstruction:** mel واقعی → BigVGAN؛ برای تست خود ووکودر
- **Generated synthesis:** mel تولیدشده توسط CFM → BigVGAN؛ برای تست کل مسیر

مدل آکوستیک هنوز در حد Smoke Test آموزش دیده، بنابراین کیفیت generated audio در این مرحله معیار نهایی نیست.

### سلول ۳۰: نصب و بارگذاری BigVGAN

برای اولین اجرا `use_cuda_kernel=False` است تا نیاز به کامپایل kernel سفارشی نباشد. بعد از اطمینان از مسیر، می‌توان گزینه CUDA را برای سرعت بیشتر آزمایش کرد.

In [ ]:
import importlib.util
import json
import subprocess
import sys

if importlib.util.find_spec("bigvgan") is None:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "git+https://github.com/NVIDIA/BigVGAN.git"],
        check=True,
    )

import bigvgan
import torch

BIGVGAN_MODEL_NAME = "nvidia/bigvgan_22khz_80band"
BIGVGAN_DEVICE = ACOUSTIC_DEVICE
BIGVGAN_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "bigvgan"
BIGVGAN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Loading vocoder:", BIGVGAN_MODEL_NAME)
vocoder = bigvgan.BigVGAN.from_pretrained(
    BIGVGAN_MODEL_NAME, use_cuda_kernel=False
)
vocoder.remove_weight_norm()
vocoder = vocoder.eval().to(BIGVGAN_DEVICE)

expected_vocoder_config = {
    "sampling_rate": MEL_CONFIG["sample_rate"],
    "num_mels": MEL_CONFIG["n_mels"],
    "n_fft": MEL_CONFIG["n_fft"],
    "hop_size": MEL_CONFIG["hop_length"],
    "win_size": MEL_CONFIG["win_length"],
    "fmin": MEL_CONFIG["f_min"],
    "fmax": MEL_CONFIG["f_max"],
}
for attribute, expected in expected_vocoder_config.items():
    actual = getattr(vocoder.h, attribute)
    assert actual == expected, f"BigVGAN {attribute}={actual}, expected {expected}"

vocoder_config_path = ACOUSTIC_MODEL_DIR / "bigvgan_config.json"
vocoder_config_path.write_text(
    json.dumps({
        "model_name": BIGVGAN_MODEL_NAME,
        "use_cuda_kernel": False,
        "expected_config": expected_vocoder_config,
        "mel_convention": "log(clamp(mel_magnitude, min=1e-5)); STFT center=False",
    }, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("BigVGAN sampling rate:", vocoder.h.sampling_rate)
print("BigVGAN mel bands:", vocoder.h.num_mels)
print("Vocoder configuration compatibility: PASSED")

### سلول ۳۱: تبدیل mel به WAV و پخش نتیجه

برای جلوگیری از تولید صوت برای padding، فقط طول معتبر نمونه به ووکودر داده می‌شود. فایل مرجع برای بررسی سلامت BigVGAN است و فایل generated نتیجه مدل آکوستیک فعلی است؛ با گوش دادن هر دو می‌توانی تفاوت را بشنوی.

In [ ]:
import json
import numpy as np
import soundfile as sf
import torch
from IPython.display import Audio, display

assert "generated_mel" in globals() and "sample_mel" in globals(), "Run the acoustic inference cell first."
assert globals().get("MEL_CONVENTION") == "bigvgan_log_mel_v1", "Rerun cells 26-29 with the BigVGAN-compatible mel pipeline first."
sample_length = int(sample_batch["lengths"][0])
reference_mel = sample_mel[:1, :, :sample_length].to(BIGVGAN_DEVICE)
synthesized_mel = generated_mel[:1, :, :sample_length].to(BIGVGAN_DEVICE)
assert reference_mel.shape[1] == vocoder.h.num_mels

with torch.inference_mode():
    reference_waveform = vocoder(reference_mel)
    generated_waveform = vocoder(synthesized_mel)


def save_vocoder_waveform(waveform, output_path):
    array = waveform.squeeze().detach().cpu().numpy()
    assert array.ndim == 1 and len(array) > 0
    assert np.isfinite(array).all()
    array = np.clip(array, -1.0, 1.0)
    sf.write(str(output_path), array, int(vocoder.h.sampling_rate), subtype="PCM_16")
    return array


reference_audio_path = BIGVGAN_OUTPUT_DIR / "reference_reconstruction.wav"
generated_audio_path = BIGVGAN_OUTPUT_DIR / "generated_synthesis.wav"
reference_array = save_vocoder_waveform(reference_waveform, reference_audio_path)
generated_array = save_vocoder_waveform(generated_waveform, generated_audio_path)

synthesis_manifest = {
    "vocoder": BIGVGAN_MODEL_NAME,
    "sample_rate": int(vocoder.h.sampling_rate),
    "reference_audio": str(reference_audio_path),
    "generated_audio": str(generated_audio_path),
    "generated_duration_seconds": len(generated_array) / int(vocoder.h.sampling_rate),
    "acoustic_checkpoint": str(acoustic_checkpoint_path),
    "note": "Acoustic model is smoke-trained; evaluate after full training.",
}
(BIGVGAN_OUTPUT_DIR / "synthesis_manifest.json").write_text(
    json.dumps(synthesis_manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

print("Reference reconstruction:", reference_audio_path)
print("Generated synthesis:", generated_audio_path)
print("Generated duration (seconds):", round(synthesis_manifest["generated_duration_seconds"], 3))
print("BigVGAN waveform synthesis: PASSED")
print("Reference audio:")
display(Audio(str(reference_audio_path)))
print("Generated audio:")
display(Audio(str(generated_audio_path)))

## گام ۳.۳ — مسیر mel تا waveform متصل شد

اکنون خروجی مدل آکوستیک می‌تواند به BigVGAN برسد و فایل WAV بسازد. کیفیت نهایی لحن و احساس به آموزش کامل مدل آکوستیک و ارزیابی شنیداری وابسته است.

گام بعدی roadmap: **فاز ۴، گام ۴.۱ — Sentence/Clause Chunking هوشمند برای کاهش تأخیر و پشتیبانی از متن‌های طولانی**.

# فاز ۴ — Real-Time Pipeline
## گام ۴.۱ — Sentence/Clause Chunking هوشمند

برای کاهش تأخیر، متن فارسی به chunkهای مستقل و مرتب تقسیم می‌شود. الگوریتم:

1. ابتدا مرز جمله را با علائم فارسی و انگلیسی پیدا می‌کند.
2. جمله‌های بلند را در مرز بند (`،`، `؛`، `;` و `:`) شکست می‌دهد.
3. اگر باز هم طولانی بود، روی فاصله بین کلمات تقسیم می‌کند و کلمه را نصف نمی‌کند.
4. قطعه‌های کوچک را تا حد امکان کنار هم قرار می‌دهد تا overhead کاهش یابد.

خروجی هر chunk دارای `chunk_id`، ترتیب، متن و وضعیت `is_final` است و در گام ۴.۲ می‌توان آن را به workerهای async فرستاد.

### سلول ۳۲: پیاده‌سازی chunker فارسی

این پیاده‌سازی rule-based و سبک است و برای شروع real-time pipeline به مدل اضافی نیاز ندارد. نیم‌فاصله فارسی حفظ می‌شود و نقطه بین ارقام اعشاری به‌عنوان پایان جمله در نظر گرفته نمی‌شود.

In [ ]:
import json
import re
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/Persian_Emotional_TTS")
CHUNKING_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "chunking"
CHUNKING_CONFIG_DIR = PROJECT_ROOT / "configs"
CHUNKING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHUNKING_CONFIG_DIR.mkdir(parents=True, exist_ok=True)

SENTENCE_ENDINGS = frozenset(".!?؟…")
CLAUSE_BOUNDARY_PATTERN = r"(?<=[،؛;:])\s+"
CLOSING_CHARACTERS = frozenset("\"'»«)]}")


def normalize_chunk_text(text):
    return re.sub(r"\s+", " ", text).strip()


def split_persian_sentences(text):
    sentences = []
    start = 0
    index = 0
    while index < len(text):
        character = text[index]
        if character not in SENTENCE_ENDINGS:
            index += 1
            continue
        if (
            character == "."
            and index > 0
            and index + 1 < len(text)
            and text[index - 1].isdigit()
            and text[index + 1].isdigit()
        ):
            index += 1
            continue

        end = index + 1
        while end < len(text) and text[end] in SENTENCE_ENDINGS:
            end += 1
        while end < len(text) and text[end] in CLOSING_CHARACTERS:
            end += 1
        sentence = text[start:end].strip()
        if sentence:
            sentences.append(sentence)
        start = end
        index = end

    tail = text[start:].strip()
    if tail:
        sentences.append(tail)
    return sentences


def split_long_segment_on_words(segment, max_chars):
    if len(segment) <= max_chars:
        return [segment]
    pieces = []
    current = ""
    for word in segment.split():
        candidate = word if not current else current + " " + word
        if current and len(candidate) > max_chars:
            pieces.append(current)
            current = word
        else:
            current = candidate
    if current:
        pieces.append(current)
    return pieces


def pack_segments(segments, max_chars, min_chars):
    packed = []
    current = ""
    for segment in segments:
        for piece in split_long_segment_on_words(segment, max_chars):
            if not current:
                current = piece
            elif len(current) + 1 + len(piece) <= max_chars:
                current += " " + piece
            else:
                packed.append(current)
                current = piece
    if current:
        packed.append(current)

    if len(packed) > 1 and len(packed[-1]) < min_chars:
        merged = packed[-2] + " " + packed[-1]
        if len(merged) <= max_chars:
            packed[-2] = merged
            packed.pop()
    return packed


def chunk_persian_text(text, max_chars=180, min_chars=40):
    """Return ordered, JSON-ready chunks without splitting words."""
    assert max_chars > 0
    min_chars = min(min_chars, max_chars)
    normalized = normalize_chunk_text(text)
    if not normalized:
        return []

    segments = []
    for sentence in split_persian_sentences(normalized):
        if len(sentence) <= max_chars:
            segments.append(sentence)
            continue
        clauses = [
            part.strip()
            for part in re.split(CLAUSE_BOUNDARY_PATTERN, sentence)
            if part.strip()
        ]
        segments.extend(clauses if len(clauses) > 1 else [sentence])

    packed = pack_segments(segments, max_chars, min_chars)
    return [
        {
            "chunk_id": f"chunk_{index:04d}",
            "index": index,
            "text": chunk,
            "char_count": len(chunk),
            "is_final": index == len(packed) - 1,
        }
        for index, chunk in enumerate(packed)
    ]

### سلول ۳۳: تست مرزها و ذخیره نمونه خروجی

تست‌ها بررسی می‌کنند که متن دوباره به همان ترتیب قابل بازیابی است، chunkها از حد مجاز فراتر نمی‌روند (به‌جز یک کلمه بسیار بلند)، نیم‌فاصله حفظ می‌شود و عدد اعشاری باعث شکست اشتباه نمی‌شود.

In [ ]:
import json

sample_story = (
    "او از خانه بیرون آمد. آسمان ابری بود، اما باران نمی‌بارید! "
    "دختری که می‌توانست قصه بخواند، با صدای آرام گفت: امروز هوا دلپذیر است؛ "
    "آیا تو هم با ما می‌آیی؟"
)
sample_chunks = chunk_persian_text(sample_story, max_chars=90, min_chars=25)
reconstructed = " ".join(chunk["text"] for chunk in sample_chunks)
assert reconstructed == normalize_chunk_text(sample_story)
assert all(
    chunk["char_count"] <= 90 or len(chunk["text"].split()) == 1
    for chunk in sample_chunks
)
assert "می‌توانست" in reconstructed

long_clause_text = (
    "او در مسیر طولانی جنگل قدم می‌زد، به صدای پرندگان گوش می‌داد، "
    "از کنار رودخانه عبور می‌کرد، و در نهایت به روستای کوچکی رسید."
)
long_chunks = chunk_persian_text(long_clause_text, max_chars=60, min_chars=15)
assert len(long_chunks) > 1
assert " ".join(chunk["text"] for chunk in long_chunks) == normalize_chunk_text(long_clause_text)

# A decimal point must not become a sentence boundary.
decimal_chunks = chunk_persian_text("مقدار پی برابر 3.14 است. این یک تست است.", max_chars=100)
assert len(decimal_chunks) == 1

chunking_config = {
    "chunker_version": "persian_rule_based_v1",
    "default_max_chars": 180,
    "default_min_chars": 40,
    "sentence_endings": sorted(SENTENCE_ENDINGS),
    "clause_boundary_regex": CLAUSE_BOUNDARY_PATTERN,
    "word_splitting": False,
}
chunking_config_path = CHUNKING_CONFIG_DIR / "chunking_config.json"
chunking_config_path.write_text(
    json.dumps(chunking_config, ensure_ascii=False, indent=2), encoding="utf-8"
)
sample_chunks_path = CHUNKING_OUTPUT_DIR / "sample_chunks.json"
sample_chunks_path.write_text(
    json.dumps(sample_chunks, ensure_ascii=False, indent=2), encoding="utf-8"
)

for chunk in sample_chunks:
    print(chunk["chunk_id"], chunk["char_count"], chunk["text"])
print("Chunking config:", chunking_config_path)
print("Sample chunks:", sample_chunks_path)
print("Persian chunking smoke test: PASSED")

## گام ۴.۱ — chunker آماده شد

خروجی `chunk_persian_text` می‌تواند مستقیماً به ورودی مرحله بعد برود. برای استفاده در پایپ‌لاین، مقدار پیش‌فرض `max_chars=180` را با توجه به تأخیر و طول جمله تنظیم می‌کنیم.

گام بعدی: **۴.۲ — طراحی Async Pipeline برای پردازش موازی chunkها و کاهش تأخیر اولیه**.

## گام ۴.۲ — Async Pipeline با حفظ ترتیب خروجی

در این گام chunkهای گام ۴.۱ به workerهای async داده می‌شوند. پایپ‌لاین:

1. حداکثر تعداد workerهای هم‌زمان را محدود می‌کند.
2. اجازه می‌دهد chunkهای بعدی زودتر آماده شوند، اما نتیجه را فقط به ترتیب `index` تحویل می‌دهد.
3. اولین نتیجهٔ آمادهٔ متوالی را بدون انتظار برای پایان همهٔ chunkها ارائه می‌کند.
4. در صورت خطا، کارهای باقی‌مانده را لغو می‌کند و خطا را با `chunk_id` مشخص گزارش می‌دهد.

worker واقعی باید یک تابع `async` باشد. برای یک تابع blocking می‌توان از adapter این گام استفاده کرد؛ با این حال، اجرای هم‌زمان مدل روی GPU باید از نظر مصرف حافظه و thread-safety جداگانه آزمایش شود. زمان‌های تست این گام مربوط به worker مصنوعی‌اند و ادعای سرعت synthesis واقعی نیستند.

### سلول ۳۴: پیاده‌سازی صف async محدود و مرتب

ورودی همان dictionaryهای خروجی `chunk_persian_text` است. متد `stream` یک async iterator برمی‌گرداند و متد `collect` همهٔ نتایج مرتب را به شکل لیست جمع می‌کند. اگر ورودی به‌اشتباه مخلوط‌شده باشد، ترتیب منطقی از روی `index` بازیابی می‌شود؛ indexها باید از صفر و بدون فاصله باشند.

In [ ]:
import asyncio
import inspect
import json
import time
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/Persian_Emotional_TTS")
ASYNC_PIPELINE_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "async_pipeline"
ASYNC_PIPELINE_CONFIG_DIR = PROJECT_ROOT / "configs"
ASYNC_PIPELINE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ASYNC_PIPELINE_CONFIG_DIR.mkdir(parents=True, exist_ok=True)

ASYNC_MAX_CONCURRENCY = 2
ASYNC_CHUNK_TIMEOUT_SECONDS = None  # Set a positive value after profiling the real worker.


class ChunkProcessingError(RuntimeError):
    """Identify the ordered chunk whose async worker failed."""

    def __init__(self, chunk, original_error):
        self.chunk_id = chunk["chunk_id"]
        self.chunk_index = chunk["index"]
        self.original_error = original_error
        super().__init__(
            f"{self.chunk_id} (index={self.chunk_index}) failed: {original_error}"
        )


class OrderedAsyncChunkPipeline:
    """Run async chunk workers concurrently but emit contiguous results in order."""

    def __init__(self, worker, max_concurrency=2, timeout_seconds=None):
        if not callable(worker):
            raise TypeError("worker must be callable")
        if type(max_concurrency) is not int or max_concurrency < 1:
            raise ValueError("max_concurrency must be a positive integer")
        if timeout_seconds is not None and timeout_seconds <= 0:
            raise ValueError("timeout_seconds must be positive or None")
        self.worker = worker
        self.max_concurrency = max_concurrency
        self.timeout_seconds = timeout_seconds

    @staticmethod
    def _prepare_chunks(chunks):
        ordered_chunks = list(chunks)
        for chunk in ordered_chunks:
            if not isinstance(chunk, dict):
                raise TypeError("Every chunk must be a dictionary")
            if type(chunk.get("index")) is not int or chunk["index"] < 0:
                raise ValueError("Every chunk needs a non-negative integer index")
            if not isinstance(chunk.get("chunk_id"), str) or not chunk["chunk_id"]:
                raise ValueError("Every chunk needs a non-empty chunk_id")
            if not isinstance(chunk.get("text"), str) or not chunk["text"].strip():
                raise ValueError("Every chunk needs non-empty text")

        ordered_chunks.sort(key=lambda chunk: chunk["index"])
        expected_indices = list(range(len(ordered_chunks)))
        actual_indices = [chunk["index"] for chunk in ordered_chunks]
        if actual_indices != expected_indices:
            raise ValueError("Chunk indices must be unique and contiguous from zero")
        if len({chunk["chunk_id"] for chunk in ordered_chunks}) != len(ordered_chunks):
            raise ValueError("chunk_id values must be unique")
        for position, chunk in enumerate(ordered_chunks):
            expected_final = position == len(ordered_chunks) - 1
            if bool(chunk.get("is_final")) != expected_final:
                raise ValueError(f"Invalid is_final marker for {chunk['chunk_id']}")
        return ordered_chunks

    async def _invoke_worker(self, chunk):
        started = time.perf_counter()
        # The shallow copy prevents a worker from replacing fields in the input dict.
        awaitable = self.worker(dict(chunk))
        if not inspect.isawaitable(awaitable):
            raise TypeError(
                "worker must return an awaitable; use wrap_blocking_worker for sync code"
            )
        if self.timeout_seconds is None:
            output = await awaitable
        else:
            output = await asyncio.wait_for(awaitable, timeout=self.timeout_seconds)
        return {
            "chunk_id": chunk["chunk_id"],
            "index": chunk["index"],
            "text": chunk["text"],
            "is_final": chunk["is_final"],
            "output": output,
            "processing_seconds": time.perf_counter() - started,
        }

    async def stream(self, chunks):
        """Yield results in chunk order; cancel outstanding work on failure/closure."""
        ordered_chunks = self._prepare_chunks(chunks)
        total = len(ordered_chunks)
        if total == 0:
            return

        tasks = {}
        ready = {}
        next_to_schedule = 0
        next_to_emit = 0
        try:
            while next_to_emit < total:
                while (
                    next_to_schedule < total
                    and len(tasks) < self.max_concurrency
                ):
                    position = next_to_schedule
                    task = asyncio.create_task(
                        self._invoke_worker(ordered_chunks[position])
                    )
                    tasks[task] = position
                    next_to_schedule += 1

                done, _ = await asyncio.wait(
                    set(tasks), return_when=asyncio.FIRST_COMPLETED
                )
                for task in done:
                    position = tasks.pop(task)
                    try:
                        ready[position] = task.result()
                    except Exception as exc:
                        chunk = ordered_chunks[position]
                        raise ChunkProcessingError(chunk, exc) from exc

                # A later chunk may be ready, but never skip an unfinished predecessor.
                while next_to_emit in ready:
                    yield ready.pop(next_to_emit)
                    next_to_emit += 1
        finally:
            for task in tasks:
                task.cancel()
            if tasks:
                await asyncio.gather(*tasks, return_exceptions=True)

    async def collect(self, chunks):
        return [result async for result in self.stream(chunks)]


def wrap_blocking_worker(blocking_worker):
    """Adapt a blocking function without blocking the notebook event loop."""
    if not callable(blocking_worker):
        raise TypeError("blocking_worker must be callable")

    async def async_worker(chunk):
        return await asyncio.to_thread(blocking_worker, chunk)

    return async_worker


async_pipeline_config = {
    "pipeline_version": "ordered_async_chunks_v1",
    "input_contract": "chunk_persian_text dictionaries with contiguous indices",
    "default_max_concurrency": ASYNC_MAX_CONCURRENCY,
    "default_timeout_seconds": ASYNC_CHUNK_TIMEOUT_SECONDS,
    "ordered_emission": True,
    "failure_policy": "cancel_pending_and_raise_ChunkProcessingError",
    "worker_contract": "async callable accepting one chunk dictionary",
    "performance_status": "requires_measurement_with_real_tts_worker",
}
async_pipeline_config_path = ASYNC_PIPELINE_CONFIG_DIR / "async_pipeline_config.json"
async_pipeline_config_path.write_text(
    json.dumps(async_pipeline_config, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Async pipeline config:", async_pipeline_config_path)
print("Default max concurrency:", ASYNC_MAX_CONCURRENCY)
print("Ordered async pipeline: READY")

### سلول ۳۵: تست هم‌زمانی، ترتیب، زمان اولین خروجی و خطا

worker این تست فقط با `asyncio.sleep` کار واقعی TTS را شبیه‌سازی می‌کند و هیچ صوتی تولید نمی‌کند. chunkها عمداً معکوس به ورودی داده می‌شوند و chunk صفر کندتر است؛ بنابراین تست بررسی می‌کند که ترتیب تحویل همچنان صفر تا n بماند. `await` در سطح سلول به‌صورت native در Colab پشتیبانی می‌شود.

In [ ]:
async def run_async_pipeline_smoke_test(chunks, max_concurrency=2):
    if len(chunks) < 4:
        raise ValueError("The smoke test needs at least four chunks")

    active_workers = 0
    peak_active_workers = 0
    completion_order = []

    async def synthetic_worker(chunk):
        nonlocal active_workers, peak_active_workers
        active_workers += 1
        peak_active_workers = max(peak_active_workers, active_workers)
        try:
            # Force out-of-order completion without doing any TTS computation.
            delay = 0.08 if chunk["index"] == 0 else 0.035
            await asyncio.sleep(delay)
            return f"synthetic-audio:{chunk['chunk_id']}"
        finally:
            active_workers -= 1
            completion_order.append(chunk["index"])

    pipeline = OrderedAsyncChunkPipeline(
        synthetic_worker, max_concurrency=max_concurrency
    )
    results = []
    emission_times = []
    started = time.perf_counter()
    time_to_first_result = None
    completed_before_first_result = None

    # Reversed input verifies that the logical chunk index, not list order, wins.
    async for result in pipeline.stream(reversed(chunks)):
        elapsed = time.perf_counter() - started
        if time_to_first_result is None:
            time_to_first_result = elapsed
            completed_before_first_result = len(completion_order)
        results.append(result)
        emission_times.append(elapsed)

    total_seconds = time.perf_counter() - started
    emission_order = [result["index"] for result in results]
    expected_order = [chunk["index"] for chunk in chunks]
    assert emission_order == expected_order
    assert [result["chunk_id"] for result in results] == [
        chunk["chunk_id"] for chunk in chunks
    ]
    assert completion_order != emission_order, "The test did not finish out of order"
    assert sorted(completion_order) == expected_order
    assert peak_active_workers == min(max_concurrency, len(chunks))
    assert completed_before_first_result < len(chunks), (
        "The first result was not streamed before all workers completed"
    )
    assert await pipeline.collect([]) == []

    async def failing_worker(chunk):
        if chunk["index"] == 1:
            await asyncio.sleep(0.005)
            raise ValueError("Intentional async smoke-test failure")
        await asyncio.sleep(0.05)
        return None

    failure_pipeline = OrderedAsyncChunkPipeline(failing_worker, max_concurrency=2)
    failure_chunks = [
        dict(chunk, is_final=chunk["index"] == 2)
        for chunk in chunks[:3]
    ]
    try:
        await failure_pipeline.collect(failure_chunks)
    except ChunkProcessingError as exc:
        assert exc.chunk_index == 1
        assert isinstance(exc.original_error, ValueError)
    else:
        raise AssertionError("A worker exception was not propagated")

    return {
        "mode": "synthetic_async_worker_no_tts_audio",
        "results": results,
        "metrics": {
            "time_to_first_ordered_result_seconds": time_to_first_result,
            "total_seconds": total_seconds,
            "emission_times_seconds": emission_times,
            "completion_order": completion_order,
            "emission_order": emission_order,
            "peak_active_workers": peak_active_workers,
            "completed_before_first_result": completed_before_first_result,
            "empty_input_test": "passed",
            "failure_propagation_test": "passed",
        },
    }


async_sample_text = (
    "باد سرد صبحگاهی میان درختان باغ می‌چرخید و برگ‌ها را با صدایی آرام به حرکت درمی‌آورد. "
    "دخترک کتاب قصه را برداشت، نگاهی به مادرش کرد و با لبخند شروع به خواندن نمود. "
    "در روستای دورافتاده مردی مهربان زندگی می‌کرد که هر شب برای کودکان از ستاره‌ها می‌گفت. "
    "او می‌گفت هر ستاره روایتگر رؤیایی است که هنوز فرصت روشن شدن پیدا نکرده است. "
    "کودکان با کنجکاوی گوش می‌دادند، سؤال می‌پرسیدند و تصور می‌کردند که آسمان نزدیک‌تر شده است!"
)
async_chunks = chunk_persian_text(async_sample_text, max_chars=75, min_chars=25)
assert len(async_chunks) >= 4, "Increase the sample length or reduce max_chars"
async_smoke = await run_async_pipeline_smoke_test(
    async_chunks, max_concurrency=ASYNC_MAX_CONCURRENCY
)

async_smoke_path = ASYNC_PIPELINE_OUTPUT_DIR / "smoke_results.json"
async_smoke_path.write_text(
    json.dumps(async_smoke, ensure_ascii=False, indent=2), encoding="utf-8"
)
for result in async_smoke["results"]:
    print(
        result["chunk_id"],
        "| index:", result["index"],
        "| output:", result["output"],
        "| worker seconds:", round(result["processing_seconds"], 4),
    )
metrics = async_smoke["metrics"]
print("Completion order:", metrics["completion_order"])
print("Ordered emission:", metrics["emission_order"])
print("Peak active workers:", metrics["peak_active_workers"])
print(
    "Time to first ordered result (seconds):",
    round(metrics["time_to_first_ordered_result_seconds"], 4),
)
print("Total synthetic run (seconds):", round(metrics["total_seconds"], 4))
print("Failure propagation and empty-input tests: PASSED")
print("Smoke results:", async_smoke_path)
print("Ordered async pipeline smoke test: PASSED")
print("Timing note: synthetic worker only; no TTS audio or real-time speedup was measured.")

## گام ۴.۲ — پایپ‌لاین async مرتب اضافه شد

خروجی آزمایشی در `outputs/async_pipeline/smoke_results.json` و قرارداد پیکربندی در `configs/async_pipeline_config.json` ذخیره می‌شود. این تست صحت scheduling، ترتیب و خطا را بررسی می‌کند؛ کاهش تأخیر synthesis واقعی باید بعداً با worker مدل و صوت واقعی اندازه‌گیری شود.

گام بعدی: **۴.۳ — ONNX و WebSocket Audio Streaming**.

# فاز ۵ — Evaluation & Fine-Tuning
## وضعیت: در انتظار تکمیل فازهای قبلی

- **گام ۵.۱:** ارزیابی ذهنی و انسانی کیفیت لحن‌ها با MOS.
- **گام ۵.۲:** بررسی پایداری لحن در جملات طولانی و رفع اشکال‌های احتمالی.